In [1]:
import sys
!{sys.executable} -m pip install lightning

In [1]:
import os

os.environ["CUDA_LAUNCH_BLOCKING"] = "1"

import lightning as pl
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import DataLoader, Dataset

In [ ]:
class YourCustomDataset(Dataset):
    """
    Custom Dataset for remote sensing data.

    Parameters:
    -----------
    data : numpy.ndarray
        Feature data (n_samples, n_features)
    labels : numpy.ndarray
        Target labels (n_samples,)
    """

    def __init__(self, data, labels):
        self.data = data
        self.labels = labels

    def __len__(self):
        return len(self.data)

    def __getitem__(self, idx):
        x = self.data[idx]
        y = self.labels[idx]
        return x, y

## Data Loading and Random Split (CNN Baseline)

Loads the four 3×3 patch .npz files from Lab 4.2, combines them into a single 
dataset of 200,000 samples, normalizes spectral values, and creates a stratified 
80/10/10 train/val/test split. This random split is used for the CNN baseline 
experiment -- note that it does not account for spatial autocorrelation.

Loads the four 3×3 patch .npz files from Lab 4.2, combines them into a single 
dataset of 200,000 samples, normalizes spectral values, and creates a stratified 
80/10/10 train/val/test split. This random split is used for the CNN baseline 
experiment -- note that it does not account for spatial autocorrelation.

In [3]:
import numpy as np
from pathlib import Path

# --- 1) Load all .npz files and combine ---
training_dir = Path("/p/scratch/training2600/TeamGudnason/training_data")
npz_files = sorted(training_dir.glob("*_data.npz"))

print(f"Found {len(npz_files)} .npz files:")
for f in npz_files:
    print(f"  {f.name}")

all_X = []
all_y = []

for npz_path in npz_files:
    data = np.load(npz_path)
    print(f"\nKeys in {npz_path.name}: {list(data.keys())}")

    if "patches" in data and "labels" in data:
        all_X.append(data["patches"])
        all_y.append(data["labels"])
    else:
        possible_X_keys = ["X", "acquisitions", "inputs", "features", "patch"]
        possible_y_keys = ["y", "targets", "labels", "corine", "target"]
        X_key = next((k for k in possible_X_keys if k in data), None)
        y_key = next((k for k in possible_y_keys if k in data), None)
        if X_key is None or y_key is None:
            raise KeyError(f"Could not find patches/labels in {npz_path.name}. Keys: {list(data.keys())}")
        all_X.append(data[X_key])
        all_y.append(data[y_key])

X = np.concatenate(all_X, axis=0)
y = np.concatenate(all_y, axis=0)

print(f"\nCombined X shape: {X.shape}, dtype={X.dtype}")
print(f"Combined y shape: {y.shape}, dtype={y.dtype}")

# --- 2) Preprocess ---
X = X.astype(np.float32) * 0.0001
X_flat = X.reshape(X.shape[0], -1)
print(f"Flattened X shape: {X_flat.shape}")
n_features = X_flat.shape[1]

# --- 3) Remap labels to contiguous 0..K-1 ---
unique_labels = np.unique(y)
label_to_idx = {int(lbl): i for i, lbl in enumerate(unique_labels)}
idx_to_label = {i: int(lbl) for i, lbl in enumerate(unique_labels)}
y = np.array([label_to_idx[int(lbl)] for lbl in y], dtype=np.int64)

print(f"\nLabel remapping (original -> index):")
for orig, idx in label_to_idx.items():
    print(f"  CORINE {orig:2d} -> class index {idx}")
print(f"Total classes: {len(unique_labels)}")

# --- 4) Split into train / val / test ---
def make_splits_train_val_test(X, y, train_frac=0.7, val_frac=0.15, test_frac=0.15, seed=42, stratify=True):
    total = train_frac + val_frac + test_frac
    assert abs(total - 1.0) < 1e-6, "train_frac + val_frac + test_frac must be 1.0"
    rng = np.random.default_rng(seed)
    n = len(y)
    if not stratify:
        idx = rng.permutation(n)
        n_train = int(n * train_frac)
        n_val = int(n * val_frac)
        train_idx = idx[:n_train]
        val_idx = idx[n_train:n_train + n_val]
        test_idx = idx[n_train + n_val:]
        return (X[train_idx], y[train_idx]), (X[val_idx], y[val_idx]), (X[test_idx], y[test_idx])
    classes, y_inv = np.unique(y, return_inverse=True)
    train_idx_all, val_idx_all, test_idx_all = [], [], []
    for c in range(len(classes)):
        cls_idx = np.where(y_inv == c)[0]
        cls_idx = rng.permutation(cls_idx)
        n_c = len(cls_idx)
        n_train_c = int(n_c * train_frac)
        n_val_c = int(n_c * val_frac)
        train_idx_all.append(cls_idx[:n_train_c])
        val_idx_all.append(cls_idx[n_train_c:n_train_c + n_val_c])
        test_idx_all.append(cls_idx[n_train_c + n_val_c:])
    train_idx = rng.permutation(np.concatenate(train_idx_all))
    val_idx = rng.permutation(np.concatenate(val_idx_all))
    test_idx = rng.permutation(np.concatenate(test_idx_all))
    return (X[train_idx], y[train_idx]), (X[val_idx], y[val_idx]), (X[test_idx], y[test_idx])

(X_train, y_train), (X_val, y_val), (X_test, y_test) = make_splits_train_val_test(
    X_flat, y, train_frac=0.8, val_frac=0.1, test_frac=0.1, seed=42, stratify=True,
)

print(f"\nTraining samples:   {X_train.shape[0]}")
print(f"Validation samples: {X_val.shape[0]}")
print(f"Test samples:       {X_test.shape[0]}")
print(f"Features/sample:    {X_train.shape[1]}")

# --- 5) Sanity check label distribution ---
def label_hist(y_arr, name, topk=10):
    vals, cnts = np.unique(y_arr, return_counts=True)
    order = np.argsort(cnts)[::-1]
    print(f"\n{name} label distribution (top {topk}):")
    for v, c in zip(vals[order][:topk], cnts[order][:topk]):
        orig = idx_to_label[int(v)]
        print(f"  idx {int(v):2d} (CORINE {orig:2d}): {int(c)}")

label_hist(y_train, "Train")
label_hist(y_val, "Val")
label_hist(y_test, "Test")

Found 4 .npz files:
  patches_S2A_MSIL2A_20180523T095031_N0500_R079_T33TYN_20230903T233859_data.npz
  patches_S2A_MSIL2A_20181020T095031_N0500_R079_T33TYN_20230728T230408_data.npz
  patches_S2B_MSIL2A_20180408T095029_N0500_R079_T33TYN_20230906T162829_data.npz
  patches_S2B_MSIL2A_20180816T095029_N0500_R079_T33TYN_20230812T013012_data.npz

Keys in patches_S2A_MSIL2A_20180523T095031_N0500_R079_T33TYN_20230903T233859_data.npz: ['patches', 'labels']

Keys in patches_S2A_MSIL2A_20181020T095031_N0500_R079_T33TYN_20230728T230408_data.npz: ['patches', 'labels']

Keys in patches_S2B_MSIL2A_20180408T095029_N0500_R079_T33TYN_20230906T162829_data.npz: ['patches', 'labels']

Keys in patches_S2B_MSIL2A_20180816T095029_N0500_R079_T33TYN_20230812T013012_data.npz: ['patches', 'labels']

Combined X shape: (200000, 3, 3, 4), dtype=float32
Combined y shape: (200000,), dtype=uint8
Flattened X shape: (200000, 36)

Label remapping (original -> index):
  CORINE  2 -> class index 0
  CORINE  3 -> class index 1

## Class Distribution Analysis

Prints the training set class distribution sorted by frequency.
CORINE 12 (arable land) dominates with ~61% of samples.
The imbalance ratio of 2,214x motivates the use of weighted sampling

In [4]:
import numpy as np
import torch
from torch.utils.data import WeightedRandomSampler

y_train_np = np.asarray(y_train, dtype=np.int64)
classes, counts = np.unique(y_train_np, return_counts=True)

print("=" * 70)
print("TRAINING SET CLASS DISTRIBUTION")
print("=" * 70)
print(f"Total samples: {len(y_train_np)}")
print(f"Number of classes: {len(classes)}\n")

# Sort by count (descending)
sort_idx = np.argsort(-counts)
print("Classes (sorted by frequency):")
print("Index | CORINE | Count  | Percentage | Balance")
print("-" * 50)
for idx in sort_idx:
    c, n = classes[idx], counts[idx]
    pct = 100 * n / len(y_train_np)
    bar = "█" * int(pct / 2)
    print(
        f"{int(c):5d} | {idx_to_label[int(c)]:6d} | {int(n):6d} | {pct:6.1f}%  | {bar}"
    )

majority_class_pct = 100 * counts.max() / len(y_train_np)
print(f"\n⚠️  Majority class represents {majority_class_pct:.1f}% of data")
print(f"    This is the baseline accuracy for a 'dumb' classifier!")

# Compute imbalance ratio
imbalance_ratio = counts.max() / counts.min()
print(f"    Imbalance ratio (max/min): {imbalance_ratio:.1f}x")

TRAINING SET CLASS DISTRIBUTION
Total samples: 159994
Number of classes: 16

Classes (sorted by frequency):
Index | CORINE | Count  | Percentage | Balance
--------------------------------------------------
    5 |     12 |  97430 |   60.9%  | ██████████████████████████████
   10 |     23 |  17443 |   10.9%  | █████
    0 |      2 |  13696 |    8.6%  | ████
    9 |     21 |  12899 |    8.1%  | ████
   14 |     40 |   5056 |    3.2%  | █
    8 |     20 |   3382 |    2.1%  | █
    6 |     15 |   3360 |    2.1%  | █
    7 |     18 |   1699 |    1.1%  | 
   13 |     35 |   1209 |    0.8%  | 
   11 |     25 |   1027 |    0.6%  | 
   12 |     29 |    934 |    0.6%  | 
    4 |     11 |    681 |    0.4%  | 
    2 |      7 |    601 |    0.4%  | 
    1 |      3 |    297 |    0.2%  | 
   15 |     41 |    236 |    0.1%  | 
    3 |     10 |     44 |    0.0%  | 

⚠️  Majority class represents 60.9% of data
    This is the baseline accuracy for a 'dumb' classifier!
    Imbalance ratio (max/min): 2214.

## Filter Minority Classes

Drops classes with fewer than 200 training samples (only CORINE 10 with 44 samples).
Rebuilds the dataset and redoes the stratified split with 15 remaining classes.

In [9]:
# Drop classes with too few samples
MIN_SAMPLES = 200

print("Classes being dropped:")
classes_to_drop = []
for c, n in zip(classes, counts):
    if n < MIN_SAMPLES:
        orig = idx_to_label[int(c)]
        print(f"  CORINE {orig} (class idx {c}): {n} samples")
        classes_to_drop.append(int(c))

# Rebuild from full dataset without dropped classes
mask_full = np.isin(y, classes_to_drop, invert=True)
X_filtered = X_flat[mask_full]
y_filtered = y[mask_full]

# Remap labels to contiguous 0..K-1 again
unique_labels = np.unique(y_filtered)
label_to_idx = {int(lbl): i for i, lbl in enumerate(unique_labels)}
idx_to_label = {i: int(lbl) for i, lbl in enumerate(unique_labels)}
y_filtered = np.array([label_to_idx[int(lbl)] for lbl in y_filtered], dtype=np.int64)

# Redo splits
(X_train, y_train), (X_val, y_val), (X_test, y_test) = make_splits_train_val_test(
    X_filtered, y_filtered,
    train_frac=0.8, val_frac=0.1, test_frac=0.1,
    seed=42, stratify=True,
)

print(f"\nAfter filtering:")
print(f"  Classes kept: {len(unique_labels)}")
print(f"  Training samples:   {X_train.shape[0]}")
print(f"  Validation samples: {X_val.shape[0]}")
print(f"  Test samples:       {X_test.shape[0]}")
label_hist(y_train, "Train (filtered)")

Classes being dropped:
  CORINE 10 (class idx 3): 44 samples

After filtering:
  Classes kept: 15
  Training samples:   159950
  Validation samples: 19989
  Test samples:       20005

Train (filtered) label distribution (top 10):
  idx  4 (CORINE  5): 97430
  idx  9 (CORINE 10): 17443
  idx  0 (CORINE  0): 13696
  idx  8 (CORINE  9): 12899
  idx 13 (CORINE 14): 5056
  idx  7 (CORINE  8): 3382
  idx  5 (CORINE  6): 3360
  idx  6 (CORINE  7): 1699
  idx 12 (CORINE 13): 1209
  idx 10 (CORINE 11): 1027


## CNN Model Architecture

Defines the ConvNet class -- a simple 2-layer CNN for 3×3 Sentinel-2 patches.
Input is a flat vector (B, 36) that is reshaped to (B, 4, 3, 3) internally.
Uses AdaptiveAvgPool, BatchNorm, and dropout. Supports class-weighted loss.

In [5]:
# Recompute classes and counts from filtered y_train
y_train_np = np.asarray(y_train, dtype=np.int64)
classes, counts = np.unique(y_train_np, return_counts=True)

import lightning.pytorch as pl
import torch
import torch.nn as nn
import torch.nn.functional as F


class ConvNet(pl.LightningModule):
    """
    Simple CNN classifier for square Sentinel-2 patches.

    Assumes flat input (B, h*w*in_channels) where spatial layout is
    channels-last: the flat was produced by X.reshape(N, -1) on an
    array of shape (N, h, w, in_channels).

    patch_size: spatial size (h = w), default 3 for 3x3 patches.
    in_channels: number of spectral bands (4 or 10).
    """

    def __init__(
        self,
        num_classes=10,
        in_channels=10,
        patch_size=3,
        lr=3e-4,
        max_epochs=60,
        class_weights=None,
    ):
        super().__init__()
        self.lr = lr
        self.in_channels = in_channels
        self.patch_size = patch_size
        self.num_classes = num_classes
        self.max_epochs = max_epochs

        # Feature extraction layers
        self.features = nn.Sequential(
            nn.Conv2d(in_channels, 64, kernel_size=3, padding=1),
            nn.BatchNorm2d(64),
            nn.ReLU(inplace=True),
            nn.Conv2d(64, 128, kernel_size=3, padding=1),
            nn.BatchNorm2d(128),
            nn.ReLU(inplace=True),
            nn.AdaptiveAvgPool2d((1, 1)),
        )

        # Classification head
        self.classifier = nn.Sequential(
            nn.Flatten(),
            nn.Linear(128, 64),
            nn.ReLU(),
            nn.Dropout(0.3),
            nn.Linear(64, num_classes),
        )

        if class_weights is not None:
            self.register_buffer("class_weights", class_weights.float())
        else:
            self.class_weights = None

    def _to_image(self, x):
        """
        Convert flat input to image format (B, C, H, W).
        Handles channels-last layout: (B, h*w*c) → (B, h, w, c) → (B, c, h, w)
        """
        x = x.float()
        h = w = self.patch_size
        c = self.in_channels
        if x.dim() == 2:
            assert (
                x.size(1) == h * w * c
            ), f"Expected {h * w * c} features, got {x.size(1)}"
            x = x.view(x.size(0), h, w, c).permute(0, 3, 1, 2).contiguous()
        return x

    def forward(self, x):
        x = self._to_image(x)
        x = self.features(x)
        return self.classifier(x)

    def _loss(self, logits, y):
        return F.cross_entropy(logits, y, weight=self.class_weights)

    def training_step(self, batch, _):
        x, y = batch
        y = y.long()
        logits = self(x)
        loss = self._loss(logits, y)
        self.log("train_loss", loss, prog_bar=True)
        return loss

    def validation_step(self, batch, _):
        x, y = batch
        y = y.long()
        logits = self(x)
        loss = self._loss(logits, y)
        acc = (logits.argmax(1) == y).float().mean()
        self.log("val_loss", loss, prog_bar=True)
        self.log("val_acc", acc, prog_bar=True)
        return loss

    def test_step(self, batch, _):
        x, y = batch
        y = y.long()
        logits = self(x)
        loss = self._loss(logits, y)
        acc = (logits.argmax(1) == y).float().mean()
        self.log("test_loss", loss, prog_bar=True)
        self.log("test_acc", acc, prog_bar=True)
        return loss

    def configure_optimizers(self):
        opt = torch.optim.AdamW(self.parameters(), lr=self.lr, weight_decay=1e-4)
        sched = torch.optim.lr_scheduler.CosineAnnealingLR(opt, T_max=self.max_epochs)
        return {"optimizer": opt, "lr_scheduler": sched}

## Sampling Strategy and Model Setup

Compares two class imbalance strategies: class-weighted loss and weighted random
sampler. The weighted sampler is selected here. Creates the CNN model and
DataLoaders for the baseline experiment with 100 epochs and batch size 256.

In [6]:
# ============================================================================
# CONFIGURATION: Choose Your Sampling Strategy
# ============================================================================

batch_size = 256  # Reduced for better class mixing with sampling
max_epochs = 100

print("\n" + "=" * 70)
print("SAMPLING STRATEGY COMPARISON")
print("=" * 70)

# Strategy 1: Inverse-frequency weights for weighted loss
print("\n1️⃣  STRATEGY 1: Class-Weighted Loss (No Resampling)")
print("-" * 70)
print("   How it works:")
print("   • Each class gets a weight = 1 / frequency")
print("   • Model sees all samples in their original distribution")
print("   • Loss contribution per class is equalized")
print("   • Pros: Simple, no data duplication")
print("   • Cons: Model still biased toward majority class")

# Compute sqrt-scaled weights (softens extreme weights)
raw_weights = np.zeros(len(classes), dtype=np.float64)
for c, n in zip(classes, counts):
    raw_weights[int(c)] = 1.0 / n

sqrt_weights = np.sqrt(raw_weights)
sqrt_weights /= sqrt_weights.sum() / len(classes)  # Normalize to mean=1

print("\n   Class weights (sqrt-scaled):")
print("   Index | CORINE | Weight")
print("   " + "-" * 35)
sort_idx = np.argsort(-sqrt_weights)
for c_idx in sort_idx[: len(classes)]:
    orig = idx_to_label[c_idx]
    print(f"   {c_idx:5d} | {orig:6d} | {sqrt_weights[c_idx]:6.2f}")

class_weights_t = torch.tensor(sqrt_weights, dtype=torch.float32)

# Strategy 2: Weighted Random Sampler
print("\n\n2️⃣  STRATEGY 2: Weighted Random Sampler (Resampling)")
print("-" * 70)
print("   How it works:")
print("   • Each sample gets weight = 1 / class_frequency")
print("   • Sampler draws batches with replacement")
print("   • Rare classes appear more often in mini-batches")
print("   • Pros: Direct oversampling, minorities get more epochs")
print("   • Cons: Duplication, may overfit on rare classes")

# Map each sample to its weight
class_weight_by_value = {int(c): float(1.0 / n) for c, n in zip(classes, counts)}
sample_weights = np.array(
    [class_weight_by_value[int(lbl)] for lbl in y_train_np], dtype=np.float64
)

sampler = WeightedRandomSampler(
    weights=torch.as_tensor(sample_weights, dtype=torch.double),
    num_samples=len(sample_weights),
    replacement=True,
)

print("   ✓ Sampler created")

# ============================================================================
# SELECT WHICH STRATEGY TO USE
# ============================================================================
print("\n\n" + "=" * 70)
print("YOUR CHOICE: Which strategy will you use?")
print("=" * 70)

strategy = "weighted_sampler"  # ← CHANGE THIS TO "weighted_sampler" TO COMPARE

if strategy == "class_weights":
    print(f"\n✅ Selected: CLASS-WEIGHTED LOSS")
    class_weights_for_loss = class_weights_t
    train_sampler = None
    train_shuffle = True
    print(f"   • Using loss weights to balance classes")
    print(f"   • DataLoader will shuffle normally")

elif strategy == "weighted_sampler":
    print(f"\n✅ Selected: WEIGHTED RANDOM SAMPLER")
    class_weights_for_loss = None
    train_sampler = sampler
    train_shuffle = False
    print(f"   • Using resampling in DataLoader")
    print(f"   • Rare classes will appear more often")

else:
    print(f"\n✅ Selected: NO WEIGHTS")
    class_weights_for_loss = None
    train_sampler = None
    train_shuffle = True
    print(f"   • DataLoader will shuffle normally")

print("\n" + "=" * 70)
print("MODEL SETUP")
print("=" * 70)

# Auto-detect architecture parameters
num_classes = len(np.unique(y_train_np))
patch_size = 3  # 3x3 patches
in_channels = X_train.shape[1] // (patch_size**2)

print(f"\n  num_classes    = {num_classes}")
print(f"  patch_size     = {patch_size}×{patch_size}")
print(f"  in_channels    = {in_channels}")
print(f"  batch_size     = {batch_size}")

# Create model
model = ConvNet(
    lr=3e-4,
    num_classes=num_classes,
    in_channels=in_channels,
    patch_size=patch_size,
    class_weights=class_weights_for_loss,
    max_epochs=max_epochs,
)

print(f"\n✓ CNN Model created")

# Create datasets
train_dataset = YourCustomDataset(X_train, y_train)
val_dataset = YourCustomDataset(X_val, y_val)
test_dataset = YourCustomDataset(X_test, y_test)

# Create dataloaders
train_dataloader = DataLoader(
    train_dataset,
    batch_size=batch_size,
    sampler=train_sampler,
    shuffle=train_shuffle,
    num_workers=2,
    pin_memory=True,
)

val_dataloader = DataLoader(
    val_dataset,
    batch_size=batch_size,
    shuffle=False,
    num_workers=2,
    pin_memory=True,
)

test_dataloader = DataLoader(
    test_dataset,
    batch_size=batch_size,
    shuffle=False,
    num_workers=2,
    pin_memory=True,
)

print(f"✓ DataLoaders created")
print(f"  • Training batches:   {len(train_dataloader)}")
print(f"  • Validation batches: {len(val_dataloader)}")
print(f"  • Test batches:       {len(test_dataloader)}")


SAMPLING STRATEGY COMPARISON

1️⃣  STRATEGY 1: Class-Weighted Loss (No Resampling)
----------------------------------------------------------------------
   How it works:
   • Each class gets a weight = 1 / frequency
   • Model sees all samples in their original distribution
   • Loss contribution per class is equalized
   • Pros: Simple, no data duplication
   • Cons: Model still biased toward majority class

   Class weights (sqrt-scaled):
   Index | CORINE | Weight
   -----------------------------------
       3 |     10 |   4.41
      15 |     41 |   1.91
       1 |      3 |   1.70
       2 |      7 |   1.19
       4 |     11 |   1.12
      12 |     29 |   0.96
      11 |     25 |   0.91
      13 |     35 |   0.84
       7 |     18 |   0.71
       6 |     15 |   0.51
       8 |     20 |   0.50
      14 |     40 |   0.41
       9 |     21 |   0.26
       0 |      2 |   0.25
      10 |     23 |   0.22
       5 |     12 |   0.09


2️⃣  STRATEGY 2: Weighted Random Sampler (Resampling)

## CNN Baseline Training (Weighted Sampler, Random Split)

Trains the CNN for 100 epochs using weighted random sampling on the random split.
Achieves 43.2% overall accuracy

In [20]:
print("\n" + "=" * 70)
print("STARTING TRAINING")
print("=" * 70)
print(f"Strategy:     {strategy}")
print(f"Max epochs:   {max_epochs}")
print(f"Batch size:   {batch_size}")
print("=" * 70 + "\n")

trainer = pl.Trainer(
    accelerator="gpu",
    devices=1,
    max_epochs=max_epochs,
    gradient_clip_val=1.0,
    log_every_n_steps=1,
    enable_progress_bar=True,
)

# Start training
trainer.fit(model, train_dataloader, val_dataloader)

print("\n" + "=" * 70)
print("TESTING")
print("=" * 70)
trainer.test(model, dataloaders=test_dataloader)

Epoch 80/99 ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╸━━ 582/625 0:00:06 • 0:00:01 88.83it/s v_num: 0.000 train_loss: 1.153    
                                                                                 val_loss: 16.130 val_acc: 0.035   

IOPub message rate exceeded.
The Jupyter server will temporarily stop sending output
to the client in order to avoid crashing it.
To change this limit, set the config variable
`--ServerApp.iopub_msg_rate_limit`.

Current values:
ServerApp.iopub_msg_rate_limit=1000.0 (msgs/sec)
ServerApp.rate_limit_window=3.0 (secs)

IOPub message rate exceeded.
The Jupyter server will temporarily stop sending output
to the client in order to avoid crashing it.
To change this limit, set the config variable
`--ServerApp.iopub_msg_rate_limit`.

Current values:
ServerApp.iopub_msg_rate_limit=1000.0 (msgs/sec)
ServerApp.rate_limit_window=3.0 (secs)

IOPub message rate exceeded.
The Jupyter server will temporarily stop sending output
to the client in order to avoid crashing it.
To change this limit, set the config variable
`--ServerApp.iopub_msg_rate_limit`.

Current values:
ServerApp.iopub_msg_rate_limit=1000.0 (msgs/sec)
ServerApp.rate_limit_window=3.0 (secs)

IOPub message rate exceeded.
The Jupyter serve

LOCAL_RANK: 0 - CUDA_VISIBLE_DEVICES: [0,1]



TESTING


Output()

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━━━━┓
┃        Test metric        ┃       DataLoader 0        ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━━━━┩
│         test_acc          │    0.43234190344810486    │
│         test_loss         │    1.5378726720809937     │
└───────────────────────────┴───────────────────────────┘

[{'test_loss': 1.5378726720809937, 'test_acc': 0.43234190344810486}]

## CNN Baseline Evaluation (Random Split)

Computes overall accuracy, balanced accuracy, macro-F1, and per-class metrics
for the CNN trained on the random split. Overall accuracy: 42.9%, balanced
accuracy: 56.7%, macro-F1: 27.7%. The gap between overall and balanced accuracy
reflects the class imbalance.

In [22]:
import numpy as np
import torch


def _collect_preds_targets(model, dataloader):
    model.eval()
    device = next(model.parameters()).device
    preds_all = []
    targets_all = []
    with torch.no_grad():
        for xb, yb in dataloader:
            xb = xb.to(device)
            logits = model(xb)
            preds = torch.argmax(logits, dim=1).detach().cpu().numpy()
            targets = yb.detach().cpu().numpy()
            preds_all.append(preds)
            targets_all.append(targets)
    y_pred = np.concatenate(preds_all).astype(np.int64)
    y_true = np.concatenate(targets_all).astype(np.int64)
    return y_true, y_pred


def _confusion_matrix(y_true, y_pred):
    labels = np.unique(np.concatenate([y_true, y_pred]))
    label_to_idx = {int(lbl): i for i, lbl in enumerate(labels)}
    cm = np.zeros((len(labels), len(labels)), dtype=np.int64)
    for t, p in zip(y_true, y_pred):
        cm[label_to_idx[int(t)], label_to_idx[int(p)]] += 1
    return labels, cm


def _imbalanced_metrics_from_cm(cm):
    tp = np.diag(cm).astype(np.float64)
    support = cm.sum(axis=1).astype(np.float64)
    pred_count = cm.sum(axis=0).astype(np.float64)

    recall = np.divide(tp, support, out=np.zeros_like(tp), where=support > 0)
    precision = np.divide(tp, pred_count, out=np.zeros_like(tp), where=pred_count > 0)
    f1 = np.divide(
        2 * precision * recall,
        precision + recall,
        out=np.zeros_like(tp),
        where=(precision + recall) > 0,
    )

    balanced_acc = recall.mean()
    macro_f1 = f1.mean()
    overall_acc = tp.sum() / cm.sum() if cm.sum() > 0 else 0.0
    return overall_acc, balanced_acc, macro_f1, recall, precision, f1


# Evaluate on validation set
y_true_val, y_pred_val = _collect_preds_targets(model, val_dataloader)
labels, cm = _confusion_matrix(y_true_val, y_pred_val)
overall_acc, balanced_acc, macro_f1, recall, precision, f1 = (
    _imbalanced_metrics_from_cm(cm)
)

vals, cnts = np.unique(y_true_val, return_counts=True)
majority_baseline = cnts.max() / cnts.sum()

pred_vals, pred_cnts = np.unique(y_pred_val, return_counts=True)
pred_order = np.argsort(pred_cnts)[::-1]

print("Validation diagnostics")
print("-" * 60)
print(f"Majority-class baseline acc: {majority_baseline:.4f}")
print(f"Overall accuracy:            {overall_acc:.4f}")
print(f"Balanced accuracy:           {balanced_acc:.4f}")
print(f"Macro-F1:                    {macro_f1:.4f}")

print("\nPredicted class distribution (top 10):")
for cls, c in zip(pred_vals[pred_order][:10], pred_cnts[pred_order][:10]):
    print(f"  class {int(cls):3d}: {int(c)}")

print("\nPer-class metrics:")
print("class | support | recall | precision | f1")
for i, cls in enumerate(labels):
    sup = int(cm[i].sum())
    print(
        f"{int(cls):5d} | {sup:7d} | {recall[i]:6.3f} | {precision[i]:9.3f} | {f1[i]:5.3f}"
    )

print("\nConfusion matrix (rows=true, cols=pred):")
print(cm)

Validation diagnostics
------------------------------------------------------------
Majority-class baseline acc: 0.6092
Overall accuracy:            0.4292
Balanced accuracy:           0.5671
Macro-F1:                    0.2772

Predicted class distribution (top 10):
  class   4: 5440
  class   9: 1843
  class   5: 1806
  class   0: 1657
  class  12: 1480
  class   8: 1434
  class   7: 1366
  class   3: 1099
  class   6: 964
  class  13: 787

Per-class metrics:
class | support | recall | precision | f1
    0 |    1712 |  0.387 |     0.400 | 0.393
    1 |      37 |  0.459 |     0.048 | 0.088
    2 |      75 |  0.760 |     0.123 | 0.212
    3 |      85 |  0.612 |     0.047 | 0.088
    4 |   12178 |  0.418 |     0.935 | 0.577
    5 |     420 |  0.826 |     0.192 | 0.312
    6 |     212 |  0.462 |     0.102 | 0.167
    7 |     422 |  0.351 |     0.108 | 0.166
    8 |    1612 |  0.220 |     0.247 | 0.232
    9 |    2180 |  0.461 |     0.545 | 0.500
   10 |     128 |  0.742 |     0.164 | 0.2

In [24]:
y_true_val, y_pred_val = _collect_preds_targets(model, test_dataloader)
labels, cm = _confusion_matrix(y_true_val, y_pred_val)
overall_acc, balanced_acc, macro_f1, recall, precision, f1 = _imbalanced_metrics_from_cm(cm)

vals, cnts = np.unique(y_true_val, return_counts=True)
majority_baseline = cnts.max() / cnts.sum()

print("Test diagnostics")
print("-" * 60)
print(f"Majority-class baseline acc: {majority_baseline:.4f}")
print(f"Overall accuracy:            {overall_acc:.4f}")
print(f"Balanced accuracy:           {balanced_acc:.4f}")
print(f"Macro-F1:                    {macro_f1:.4f}")
print("\nPer-class metrics:")
print("class | support | recall | precision | f1")
for i, cls in enumerate(labels):
    sup = int(cm[i].sum())
    print(f"{int(cls):5d} | {sup:7d} | {recall[i]:6.3f} | {precision[i]:9.3f} | {f1[i]:5.3f}")

Test diagnostics
------------------------------------------------------------
Majority-class baseline acc: 0.6088
Overall accuracy:            0.4319
Balanced accuracy:           0.5713
Macro-F1:                    0.2811

Per-class metrics:
class | support | recall | precision | f1
    0 |    1712 |  0.366 |     0.397 | 0.381
    1 |      38 |  0.632 |     0.068 | 0.123
    2 |      76 |  0.671 |     0.116 | 0.198
    3 |      86 |  0.651 |     0.051 | 0.094
    4 |   12180 |  0.423 |     0.932 | 0.582
    5 |     420 |  0.874 |     0.197 | 0.322
    6 |     213 |  0.451 |     0.091 | 0.152
    7 |     424 |  0.358 |     0.113 | 0.172
    8 |    1613 |  0.206 |     0.241 | 0.222
    9 |    2181 |  0.466 |     0.535 | 0.498
   10 |     129 |  0.674 |     0.155 | 0.252
   11 |     118 |  0.466 |     0.122 | 0.194
   12 |     152 |  0.796 |     0.082 | 0.148
   13 |     632 |  0.761 |     0.625 | 0.687
   14 |      31 |  0.774 |     0.109 | 0.191


In [ ]:
# ============================================================
# RUN 2: Class weights strategy
# ============================================================

## CNN Baseline Run 2: Class Weights Strategy (Random Split)

Trains a second CNN using class-weighted loss instead of weighted sampler.
Achieves 67.1% overall accuracy -- higher than weighted sampler (43.2%) but
balanced accuracy is only 32.1%, indicating the model still predicts the
majority class most of the time. 

In [11]:
# ---- Run 2: Class weights strategy ----
strategy = "class_weights"
# ---- Run 2: Class weights strategy ----

# Recompute class weights
y_train_np = np.asarray(y_train, dtype=np.int64)
classes, counts = np.unique(y_train_np, return_counts=True)

raw_weights = np.zeros(len(classes), dtype=np.float64)
for c, n in zip(classes, counts):
    raw_weights[int(c)] = 1.0 / n
sqrt_weights = np.sqrt(raw_weights)
sqrt_weights /= sqrt_weights.sum() / len(classes)
class_weights_t = torch.tensor(sqrt_weights, dtype=torch.float32)
class_weights_for_loss = class_weights_t
train_sampler = None
train_shuffle = True

model2 = ConvNet(
    lr=3e-4,
    num_classes=num_classes,
    in_channels=in_channels,
    patch_size=patch_size,
    class_weights=class_weights_for_loss,
    max_epochs=max_epochs,
)

train_dataloader2 = DataLoader(
    train_dataset, batch_size=256, sampler=None,
    shuffle=True, num_workers=8, pin_memory=True, persistent_workers=True,
)

trainer2 = pl.Trainer(
    accelerator="gpu", devices=1, max_epochs=max_epochs,
    gradient_clip_val=1.0, log_every_n_steps=50, enable_progress_bar=True,  # <- breyta í True
)

trainer2.fit(model2, train_dataloader2, val_dataloader)
trainer2.test(model2, dataloaders=test_dataloader)

# Metrics
y_true2, y_pred2 = _collect_preds_targets(model2, test_dataloader)
labels2, cm2 = _confusion_matrix(y_true2, y_pred2)
oa2, ba2, f12, recall2, precision2, f1_per2 = _imbalanced_metrics_from_cm(cm2)

print("\n========== COMPARISON ==========")
print(f"{'Metric':<25} {'Weighted Sampler':>18} {'Class Weights':>15}")
print("-" * 60)
print(f"{'Overall accuracy':<25} {overall_acc:>18.4f} {oa2:>15.4f}")
print(f"{'Balanced accuracy':<25} {balanced_acc:>18.4f} {ba2:>15.4f}")
print(f"{'Macro-F1':<25} {macro_f1:>18.4f} {f12:>15.4f}")

Epoch 11/99 ━━━━━━━━━━━━━━━━━━━╸━━━━━━━━━━━━ 390/625 0:00:03 • 0:00:03 103.62it/s v_num: 4.000 train_loss: 1.800   
                                                                                  val_loss: 155.664 val_acc: 0.032 

IOPub message rate exceeded.
The Jupyter server will temporarily stop sending output
to the client in order to avoid crashing it.
To change this limit, set the config variable
`--ServerApp.iopub_msg_rate_limit`.

Current values:
ServerApp.iopub_msg_rate_limit=1000.0 (msgs/sec)
ServerApp.rate_limit_window=3.0 (secs)

IOPub message rate exceeded.
The Jupyter server will temporarily stop sending output
to the client in order to avoid crashing it.
To change this limit, set the config variable
`--ServerApp.iopub_msg_rate_limit`.

Current values:
ServerApp.iopub_msg_rate_limit=1000.0 (msgs/sec)
ServerApp.rate_limit_window=3.0 (secs)

IOPub message rate exceeded.
The Jupyter server will temporarily stop sending output
to the client in order to avoid crashing it.
To change this limit, set the config variable
`--ServerApp.iopub_msg_rate_limit`.

Current values:
ServerApp.iopub_msg_rate_limit=1000.0 (msgs/sec)
ServerApp.rate_limit_window=3.0 (secs)

IOPub message rate exceeded.
The Jupyter serve

LOCAL_RANK: 0 - CUDA_VISIBLE_DEVICES: [0,1,2,3     ]
SLURM auto-requeueing enabled. Setting signal handlers.


Output()

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━━━━┓
┃        Test metric        ┃       DataLoader 0        ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━━━━┩
│         test_acc          │    0.6706476211547852     │
│         test_loss         │    1.3379902839660645     │
└───────────────────────────┴───────────────────────────┘

NameError: name '_collect_preds_targets' is not defined

In [14]:
y_true2, y_pred2 = _collect_preds_targets(model2, test_dataloader)
labels2, cm2 = _confusion_matrix(y_true2, y_pred2)
oa2, ba2, f12, recall2, precision2, f1_per2 = _imbalanced_metrics_from_cm(cm2)

# Run 1 results (weighted sampler, from previous session)
overall_acc = 0.4292
balanced_acc = 0.5671
macro_f1 = 0.2772

print("\n========== COMPARISON ==========")
print(f"{'Metric':<25} {'Weighted Sampler':>18} {'Class Weights':>15}")
print("-" * 60)
print(f"{'Overall accuracy':<25} {overall_acc:>18.4f} {oa2:>15.4f}")
print(f"{'Balanced accuracy':<25} {balanced_acc:>18.4f} {ba2:>15.4f}")
print(f"{'Macro-F1':<25} {macro_f1:>18.4f} {f12:>15.4f}")
print("\n========== COMPARISON ==========")
print(f"{'Metric':<25} {'Weighted Sampler':>18} {'Class Weights':>15}")
print("-" * 60)
print(f"{'Overall accuracy':<25} {overall_acc:>18.4f} {oa2:>15.4f}")
print(f"{'Balanced accuracy':<25} {balanced_acc:>18.4f} {ba2:>15.4f}")
print(f"{'Macro-F1':<25} {macro_f1:>18.4f} {f12:>15.4f}")


========== COMPARISON ==========
Metric                      Weighted Sampler   Class Weights
------------------------------------------------------------
Overall accuracy                      0.4292          0.6708
Balanced accuracy                     0.5671          0.3208
Macro-F1                              0.2772          0.3432

========== COMPARISON ==========
Metric                      Weighted Sampler   Class Weights
------------------------------------------------------------
Overall accuracy                      0.4292          0.6708
Balanced accuracy                     0.5671          0.3208
Macro-F1                              0.2772          0.3432


In [16]:
import json
import os

os.makedirs("/p/scratch/training2600/TeamGudnason/results", exist_ok=True)

cnn_results = {
    "weighted_sampler": {
        "overall_acc": 0.4292,
        "balanced_acc": 0.5671,
        "macro_f1": 0.2772
    },
    "class_weights": {
        "overall_acc": 0.6708,
        "balanced_acc": 0.3208,
        "macro_f1": 0.3432
    }
}

with open("/p/scratch/training2600/TeamGudnason/results/cnn_results.json", "w") as f:
    json.dump(cnn_results, f, indent=2)

print("Vistað.")

Vistað.


## Horizontal Strip Split Experiment

CNN trained on horizontal strip split (upper 50% train, lower 50% test)
as in Checkpoint 2. Abandoned due to cloud cover mismatch between splits.

In [13]:
# ---- Run 2: Class weights strategy ----
strategy = "class_weights"
# ---- Run 2: Class weights strategy ----
max_epochs = 200  # bæta við fremst í cell
import numpy as np
# Recompute class weights
y_train_np = np.asarray(y_train, dtype=np.int64)
classes, counts = np.unique(y_train_np, return_counts=True)

raw_weights = np.zeros(len(classes), dtype=np.float64)
for c, n in zip(classes, counts):
    raw_weights[int(c)] = 1.0 / n
sqrt_weights = np.sqrt(raw_weights)
sqrt_weights /= sqrt_weights.sum() / len(classes)
class_weights_t = torch.tensor(sqrt_weights, dtype=torch.float32)
class_weights_for_loss = class_weights_t
train_sampler = None
train_shuffle = True

model2 = ConvNet(
    lr=3e-4,
    num_classes=num_classes,
    in_channels=in_channels,
    patch_size=patch_size,
    class_weights=class_weights_for_loss,
    max_epochs=200,  # breyta hér
)

train_dataloader2 = DataLoader(
    train_dataset, batch_size=256, sampler=None,
    shuffle=True, num_workers=8, pin_memory=True, persistent_workers=True,
)

early_stop = pl.callbacks.EarlyStopping(
    monitor="val_loss",
    patience=20,
    mode="min",
)

trainer2 = pl.Trainer(
    accelerator="gpu", devices=1, max_epochs=200,
    gradient_clip_val=1.0, log_every_n_steps=50,
    enable_progress_bar=True,
    callbacks=[early_stop],
)

trainer2.fit(model2, train_dataloader2, val_dataloader)
trainer2.test(model2, dataloaders=test_dataloader)

# Metrics
y_true2, y_pred2 = _collect_preds_targets(model2, test_dataloader)
labels2, cm2 = _confusion_matrix(y_true2, y_pred2)
oa2, ba2, f12, recall2, precision2, f1_per2 = _imbalanced_metrics_from_cm(cm2)

print("\n========== COMPARISON ==========")
print(f"{'Metric':<25} {'Weighted Sampler':>18} {'Class Weights':>15}")
print("-" * 60)
print(f"{'Overall accuracy':<25} {overall_acc:>18.4f} {oa2:>15.4f}")
print(f"{'Balanced accuracy':<25} {balanced_acc:>18.4f} {ba2:>15.4f}")
print(f"{'Macro-F1':<25} {macro_f1:>18.4f} {f12:>15.4f}")

GPU available: True (cuda), used: True
TPU available: False, using: 0 TPU cores
💡 Tip: For seamless cloud logging and experiment tracking, try installing [litlogger](https://pypi.org/project/litlogger/) to enable LitLogger, which logs metrics and artifacts automatically to the Lightning Experiments platform.
💡 Tip: For seamless cloud uploads and versioning, try installing [litmodels](https://pypi.org/project/litmodels/) to enable LitModelCheckpoint, which syncs automatically with the Lightning model registry.
You are using a CUDA device ('NVIDIA A100-SXM4-40GB') that has Tensor Cores. To properly utilize them, you should set `torch.set_float32_matmul_precision('medium' | 'high')` which will trade-off precision for performance. For more details, read https://pytorch.org/docs/stable/generated/torch.set_float32_matmul_precision.html#torch.set_float32_matmul_precision
/p/project1/training2600/TeamGudnason/envs/danube/lib/python3.12/site-packages/lightning/pytorch/callbacks/model_checkpoint

┏━━━┳━━━━━━━━━━━━┳━━━━━━━━━━━━┳━━━━━━━━┳━━━━━━━┳━━━━━━━┓
┃   ┃ Name       ┃ Type       ┃ Params ┃ Mode  ┃ FLOPs ┃
┡━━━╇━━━━━━━━━━━━╇━━━━━━━━━━━━╇━━━━━━━━╇━━━━━━━╇━━━━━━━┩
│ 0 │ features   │ Sequential │ 76.6 K │ train │     0 │
│ 1 │ classifier │ Sequential │  9.2 K │ train │     0 │
└───┴────────────┴────────────┴────────┴───────┴───────┘

Trainable params: 85.8 K                                                                                           
Non-trainable params: 0                                                                                            
Total params: 85.8 K                                                                                               
Total estimated model params size (MB): 0                                                                          
Modules in train mode: 14                                                                                          
Modules in eval mode: 0                                                                                            
Total FLOPs: 0

SLURM auto-requeueing enabled. Setting signal handlers.


Output()

...

LOCAL_RANK: 0 - CUDA_VISIBLE_DEVICES: [0,1,2,3     ]
SLURM auto-requeueing enabled. Setting signal handlers.


Output()

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━━━━┓
┃        Test metric        ┃       DataLoader 0        ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━━━━┩
│         test_acc          │    0.08557860553264618    │
│         test_loss         │     178.6322479248047     │
└───────────────────────────┴───────────────────────────┘


========== COMPARISON ==========
Metric                      Weighted Sampler   Class Weights
------------------------------------------------------------


NameError: name 'overall_acc' is not defined

## Data Preprocessing (Horizontal Split)

Loads the horizontal split data, selects first 4 spectral bands, normalizes,
remaps labels, and drops classes with fewer than 200 samples. Results in
11 classes and ~19k training samples.

In [11]:
# Taka fyrstu 4 bönd
X_train_raw = X_train_raw[:, :, :, :4]
X_val_raw   = X_val_raw[:, :, :, :4]
X_test_raw  = X_test_raw[:, :, :, :4]

# Preprocess
X_train = X_train_raw.astype(np.float32) * 0.0001
X_val   = X_val_raw.astype(np.float32) * 0.0001
X_test  = X_test_raw.astype(np.float32) * 0.0001

X_train = X_train.reshape(X_train.shape[0], -1)
X_val   = X_val.reshape(X_val.shape[0], -1)
X_test  = X_test.reshape(X_test.shape[0], -1)

# Remap labels
unique_labels = np.unique(y_train_raw)
label_to_idx = {int(lbl): i for i, lbl in enumerate(unique_labels)}
idx_to_label = {i: int(lbl) for i, lbl in enumerate(unique_labels)}

y_train = np.array([label_to_idx[int(l)] for l in y_train_raw], dtype=np.int64)
y_val   = np.array([label_to_idx.get(int(l), -1) for l in y_val_raw], dtype=np.int64)
y_test  = np.array([label_to_idx.get(int(l), -1) for l in y_test_raw], dtype=np.int64)

# Fjarlægja -1 labels (klassar ekki í train)
y_val_mask  = y_val != -1
y_test_mask = y_test != -1
X_val,  y_val  = X_val[y_val_mask],   y_val[y_val_mask]
X_test, y_test = X_test[y_test_mask], y_test[y_test_mask]

# Droppa klassa með < 200 samples
classes, counts = np.unique(y_train, return_counts=True)
valid = set(classes[counts >= 200])
mask = np.isin(y_train, list(valid))
X_train, y_train = X_train[mask], y_train[mask]
X_val  = X_val[np.isin(y_val, list(valid))]
y_val  = y_val[np.isin(y_val, list(valid))]
X_test = X_test[np.isin(y_test, list(valid))]
y_test = y_test[np.isin(y_test, list(valid))]

num_classes = len(np.unique(y_train))
print(f"Train: {X_train.shape[0]} samples, {num_classes} classes")
print(f"Val:   {X_val.shape[0]} samples")
print(f"Test:  {X_test.shape[0]} samples")

Train: 18981 samples, 11 classes
Val:   8969 samples
Test:  8357 samples


## CNN with Geographic Split (Horizontal, 3×3 patches)

Trains CNN on horizontal strip split with class weights and early stopping.
Achieves 34.6% overall accuracy -- lower than random split (67.1%) due to
spatial generalization challenge. Balanced accuracy was only 8.9%, showing
the model still predicts majority class under geographic evaluation

In [2]:
import os
os.environ["CUDA_LAUNCH_BLOCKING"] = "1"
os.environ["CUDA_VISIBLE_DEVICES"] = "0"

import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
import lightning.pytorch as pl
from torch.utils.data import DataLoader, Dataset
from pathlib import Path

# ============================================================
# Dataset og Model definitions
# ============================================================
class YourCustomDataset(Dataset):
    def __init__(self, data, labels):
        self.data = data
        self.labels = labels
    def __len__(self):
        return len(self.data)
    def __getitem__(self, idx):
        return self.data[idx], self.labels[idx]

class ConvNet(pl.LightningModule):
    def __init__(self, num_classes=10, in_channels=4, patch_size=3,
                 lr=3e-4, max_epochs=200, class_weights=None):
        super().__init__()
        self.lr = lr
        self.in_channels = in_channels
        self.patch_size = patch_size
        self.num_classes = num_classes
        self.max_epochs = max_epochs
        self.features = nn.Sequential(
            nn.Conv2d(in_channels, 64, kernel_size=3, padding=1),
            nn.BatchNorm2d(64), nn.ReLU(inplace=True),
            nn.Conv2d(64, 128, kernel_size=3, padding=1),
            nn.BatchNorm2d(128), nn.ReLU(inplace=True),
            nn.AdaptiveAvgPool2d((1, 1)),
        )
        self.classifier = nn.Sequential(
            nn.Flatten(), nn.Linear(128, 64),
            nn.ReLU(), nn.Dropout(0.3),
            nn.Linear(64, num_classes),
        )
        if class_weights is not None:
            self.register_buffer("class_weights", class_weights.float())
        else:
            self.class_weights = None

    def _to_image(self, x):
        x = x.float()
        h = w = self.patch_size
        c = self.in_channels
        if x.dim() == 2:
            x = x.view(x.size(0), h, w, c).permute(0, 3, 1, 2).contiguous()
        return x

    def forward(self, x):
        return self.classifier(self.features(self._to_image(x)))

    def _loss(self, logits, y):
        return F.cross_entropy(logits, y, weight=self.class_weights)

    def training_step(self, batch, _):
        x, y = batch
        loss = self._loss(self(x), y.long())
        self.log("train_loss", loss, prog_bar=True)
        return loss

    def validation_step(self, batch, _):
        x, y = batch
        logits = self(x)
        loss = self._loss(logits, y.long())
        acc = (logits.argmax(1) == y).float().mean()
        self.log("val_loss", loss, prog_bar=True)
        self.log("val_acc", acc, prog_bar=True)

    def test_step(self, batch, _):
        x, y = batch
        logits = self(x)
        loss = self._loss(logits, y.long())
        acc = (logits.argmax(1) == y).float().mean()
        self.log("test_loss", loss)
        self.log("test_acc", acc, prog_bar=True)

    def configure_optimizers(self):
        opt = torch.optim.AdamW(self.parameters(), lr=self.lr, weight_decay=1e-4)
        sched = torch.optim.lr_scheduler.CosineAnnealingLR(opt, T_max=self.max_epochs)
        return {"optimizer": opt, "lr_scheduler": sched}

# ============================================================
# Load data
# ============================================================
FINAL_DIR = Path("/p/scratch/training2600/TeamGudnason/training_data/final_geo")

X_train_raw = np.load(FINAL_DIR / "patches_train.npz")['patches'][:, :, :, :4]
y_train_raw = np.load(FINAL_DIR / "labels_train.npy")
X_val_raw   = np.load(FINAL_DIR / "patches_val.npz")['patches'][:, :, :, :4]
y_val_raw   = np.load(FINAL_DIR / "labels_val.npy")
X_test_raw  = np.load(FINAL_DIR / "patches_test.npz")['patches'][:, :, :, :4]
y_test_raw  = np.load(FINAL_DIR / "labels_test.npy")

# Normalize
X_train = (X_train_raw.astype(np.float32) * 0.0001).reshape(X_train_raw.shape[0], -1)
X_val   = (X_val_raw.astype(np.float32)   * 0.0001).reshape(X_val_raw.shape[0], -1)
X_test  = (X_test_raw.astype(np.float32)  * 0.0001).reshape(X_test_raw.shape[0], -1)

# Drop classes with < 200 samples
classes_tr, counts_tr = np.unique(y_train_raw, return_counts=True)
valid_corine = set(classes_tr[counts_tr >= 200])
mask_tr = np.isin(y_train_raw, list(valid_corine))
mask_va = np.isin(y_val_raw,   list(valid_corine))
mask_te = np.isin(y_test_raw,  list(valid_corine))
X_train, y_tr = X_train[mask_tr], y_train_raw[mask_tr]
X_val,   y_va = X_val[mask_va],   y_val_raw[mask_va]
X_test,  y_te = X_test[mask_te],  y_test_raw[mask_te]

# Remap labels to 0..K-1
unique_labels = np.unique(y_tr)
label_to_idx  = {int(lbl): i for i, lbl in enumerate(unique_labels)}
idx_to_label  = {i: int(lbl) for i, lbl in enumerate(unique_labels)}
y_train = np.array([label_to_idx[int(l)] for l in y_tr], dtype=np.int64)
y_val   = np.array([label_to_idx.get(int(l), -1) for l in y_va], dtype=np.int64)
y_test  = np.array([label_to_idx.get(int(l), -1) for l in y_te], dtype=np.int64)

# Remove unseen labels
X_val,  y_val  = X_val[y_val != -1],   y_val[y_val != -1]
X_test, y_test = X_test[y_test != -1], y_test[y_test != -1]

num_classes = len(unique_labels)
print(f"Train: {X_train.shape[0]} samples, {num_classes} classes")
print(f"Val:   {X_val.shape[0]} samples")
print(f"Test:  {X_test.shape[0]} samples")
print(f"y_train range: {y_train.min()} - {y_train.max()}")
print(f"y_val range:   {y_val.min()} - {y_val.max()}")

# ============================================================
# Class weights
# ============================================================
y_train_np = np.asarray(y_train, dtype=np.int64)
classes, counts = np.unique(y_train_np, return_counts=True)
raw_weights = np.zeros(len(classes), dtype=np.float64)
for c, n in zip(classes, counts):
    raw_weights[int(c)] = 1.0 / n
sqrt_weights = np.sqrt(raw_weights)
sqrt_weights /= sqrt_weights.sum() / len(classes)
class_weights_t = torch.tensor(sqrt_weights, dtype=torch.float32)

# ============================================================
# Dataloaders
# ============================================================
train_dataset = YourCustomDataset(X_train, y_train)
val_dataset   = YourCustomDataset(X_val,   y_val)
test_dataset  = YourCustomDataset(X_test,  y_test)

train_dataloader = DataLoader(train_dataset, batch_size=256, shuffle=True,
                               num_workers=4, pin_memory=False)
val_dataloader   = DataLoader(val_dataset,   batch_size=256, shuffle=False,
                               num_workers=4, pin_memory=False)
test_dataloader  = DataLoader(test_dataset,  batch_size=256, shuffle=False,
                               num_workers=4, pin_memory=False)

# ============================================================
# Model + Trainer
# ============================================================
patch_size  = 3
in_channels = X_train.shape[1] // (patch_size**2)
max_epochs  = 200

model_geo = ConvNet(
    lr=3e-4, num_classes=num_classes, in_channels=in_channels,
    patch_size=patch_size, class_weights=class_weights_t, max_epochs=max_epochs,
)

early_stop = pl.callbacks.EarlyStopping(monitor="val_loss", patience=20, mode="min")

trainer_geo = pl.Trainer(
    accelerator="gpu", devices=1, max_epochs=max_epochs,
    gradient_clip_val=1.0, log_every_n_steps=50,
    enable_progress_bar=True, callbacks=[early_stop],
)

trainer_geo.fit(model_geo, train_dataloader, val_dataloader)
trainer_geo.test(model_geo, dataloaders=test_dataloader)

GPU available: True (cuda), used: True
TPU available: False, using: 0 TPU cores
💡 Tip: For seamless cloud logging and experiment tracking, try installing [litlogger](https://pypi.org/project/litlogger/) to enable LitLogger, which logs metrics and artifacts automatically to the Lightning Experiments platform.
💡 Tip: For seamless cloud uploads and versioning, try installing [litmodels](https://pypi.org/project/litmodels/) to enable LitModelCheckpoint, which syncs automatically with the Lightning model registry.
/p/project1/training2600/TeamGudnason/envs/danube/lib/python3.12/site-packages/lightning/pytorch/callbacks/model_checkpoint.py:881: Checkpoint directory /p/project1/training2600/gudnason2/repos/EO_danube_river/lightning_logs/version_14664570/checkpoints exists and is not empty.
LOCAL_RANK: 0 - CUDA_VISIBLE_DEVICES: [0]


Train: 18981 samples, 11 classes
Val:   8969 samples
Test:  8357 samples
y_train range: 0 - 10
y_val range:   0 - 10


┏━━━┳━━━━━━━━━━━━┳━━━━━━━━━━━━┳━━━━━━━━┳━━━━━━━┳━━━━━━━┓
┃   ┃ Name       ┃ Type       ┃ Params ┃ Mode  ┃ FLOPs ┃
┡━━━╇━━━━━━━━━━━━╇━━━━━━━━━━━━╇━━━━━━━━╇━━━━━━━╇━━━━━━━┩
│ 0 │ features   │ Sequential │ 76.6 K │ train │     0 │
│ 1 │ classifier │ Sequential │  9.0 K │ train │     0 │
└───┴────────────┴────────────┴────────┴───────┴───────┘

Trainable params: 85.6 K                                                                                           
Non-trainable params: 0                                                                                            
Total params: 85.6 K                                                                                               
Total estimated model params size (MB): 0                                                                          
Modules in train mode: 14                                                                                          
Modules in eval mode: 0                                                                                            
Total FLOPs: 0

SLURM auto-requeueing enabled. Setting signal handlers.


Output()

LOCAL_RANK: 0 - CUDA_VISIBLE_DEVICES: [0]
SLURM auto-requeueing enabled. Setting signal handlers.


Output()

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━━━━┓
┃        Test metric        ┃       DataLoader 0        ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━━━━┩
│         test_acc          │    0.34569820761680603    │
│         test_loss         │    13.314289093017578     │
└───────────────────────────┴───────────────────────────┘

[{'test_loss': 13.314289093017578, 'test_acc': 0.34569820761680603}]

In [3]:
def _collect_preds_targets(model, dataloader):
    model.eval()
    device = next(model.parameters()).device
    preds_all, targets_all = [], []
    with torch.no_grad():
        for xb, yb in dataloader:
            xb = xb.to(device)
            preds_all.append(torch.argmax(model(xb), dim=1).cpu().numpy())
            targets_all.append(yb.cpu().numpy())
    return np.concatenate(preds_all).astype(np.int64), np.concatenate(targets_all).astype(np.int64)

def _confusion_matrix(y_true, y_pred):
    labels = np.unique(np.concatenate([y_true, y_pred]))
    label_to_idx = {int(lbl): i for i, lbl in enumerate(labels)}
    cm = np.zeros((len(labels), len(labels)), dtype=np.int64)
    for t, p in zip(y_true, y_pred):
        cm[label_to_idx[int(t)], label_to_idx[int(p)]] += 1
    return labels, cm

def _imbalanced_metrics_from_cm(cm):
    tp = np.diag(cm).astype(np.float64)
    support = cm.sum(axis=1).astype(np.float64)
    pred_count = cm.sum(axis=0).astype(np.float64)
    recall = np.divide(tp, support, out=np.zeros_like(tp), where=support > 0)
    precision = np.divide(tp, pred_count, out=np.zeros_like(tp), where=pred_count > 0)
    f1 = np.divide(2*precision*recall, precision+recall, out=np.zeros_like(tp), where=(precision+recall) > 0)
    return tp.sum()/cm.sum(), recall.mean(), f1.mean(), recall, precision, f1

y_true, y_pred = _collect_preds_targets(model_geo, test_dataloader)
labels, cm = _confusion_matrix(y_true, y_pred)
oa, ba, mf1, recall, precision, f1 = _imbalanced_metrics_from_cm(cm)

print(f"Overall accuracy:  {oa:.4f}")
print(f"Balanced accuracy: {ba:.4f}")
print(f"Macro-F1:          {mf1:.4f}")

Overall accuracy:  0.3457
Balanced accuracy: 0.0888
Macro-F1:          0.0694


In [5]:
import os
os.environ["CUDA_LAUNCH_BLOCKING"] = "1"
os.environ["CUDA_VISIBLE_DEVICES"] = "0"

import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
import lightning.pytorch as pl
from torch.utils.data import DataLoader, Dataset
from pathlib import Path

class YourCustomDataset(Dataset):
    def __init__(self, data, labels):
        self.data = data
        self.labels = labels
    def __len__(self):
        return len(self.data)
    def __getitem__(self, idx):
        return self.data[idx], self.labels[idx]

class ConvNet(pl.LightningModule):
    def __init__(self, num_classes=5, in_channels=4, patch_size=3,
                 lr=3e-4, max_epochs=200, class_weights=None):
        super().__init__()
        self.lr = lr
        self.in_channels = in_channels
        self.patch_size = patch_size
        self.num_classes = num_classes
        self.max_epochs = max_epochs
        self.features = nn.Sequential(
            nn.Conv2d(in_channels, 64, kernel_size=3, padding=1),
            nn.BatchNorm2d(64), nn.ReLU(inplace=True),
            nn.Conv2d(64, 128, kernel_size=3, padding=1),
            nn.BatchNorm2d(128), nn.ReLU(inplace=True),
            nn.AdaptiveAvgPool2d((1, 1)),
        )
        self.classifier = nn.Sequential(
            nn.Flatten(), nn.Linear(128, 64),
            nn.ReLU(), nn.Dropout(0.3),
            nn.Linear(64, num_classes),
        )
        if class_weights is not None:
            self.register_buffer("class_weights", class_weights.float())
        else:
            self.class_weights = None

    def _to_image(self, x):
        x = x.float()
        h = w = self.patch_size
        c = self.in_channels
        if x.dim() == 2:
            x = x.view(x.size(0), h, w, c).permute(0, 3, 1, 2).contiguous()
        return x

    def forward(self, x):
        return self.classifier(self.features(self._to_image(x)))

    def _loss(self, logits, y):
        return F.cross_entropy(logits, y, weight=self.class_weights)

    def training_step(self, batch, _):
        x, y = batch
        loss = self._loss(self(x), y.long())
        self.log("train_loss", loss, prog_bar=True)
        return loss

    def validation_step(self, batch, _):
        x, y = batch
        logits = self(x)
        loss = self._loss(logits, y.long())
        acc = (logits.argmax(1) == y).float().mean()
        self.log("val_loss", loss, prog_bar=True)
        self.log("val_acc", acc, prog_bar=True)

    def test_step(self, batch, _):
        x, y = batch
        logits = self(x)
        loss = self._loss(logits, y.long())
        acc = (logits.argmax(1) == y).float().mean()
        self.log("test_loss", loss)
        self.log("test_acc", acc, prog_bar=True)

    def configure_optimizers(self):
        opt = torch.optim.AdamW(self.parameters(), lr=self.lr, weight_decay=1e-4)
        sched = torch.optim.lr_scheduler.CosineAnnealingLR(opt, T_max=self.max_epochs)
        return {"optimizer": opt, "lr_scheduler": sched}

# Load data
FINAL_DIR = Path("/p/scratch/training2600/TeamGudnason/training_data/final_geo_l1")

X_train_raw = np.load(FINAL_DIR / "patches_train.npz")['patches']
y_train     = np.load(FINAL_DIR / "labels_train.npy").astype(np.int64)
X_val_raw   = np.load(FINAL_DIR / "patches_val.npz")['patches']
y_val       = np.load(FINAL_DIR / "labels_val.npy").astype(np.int64)
X_test_raw  = np.load(FINAL_DIR / "patches_test.npz")['patches']
y_test      = np.load(FINAL_DIR / "labels_test.npy").astype(np.int64)

X_train = (X_train_raw.astype(np.float32) * 0.0001).reshape(X_train_raw.shape[0], -1)
X_val   = (X_val_raw.astype(np.float32)   * 0.0001).reshape(X_val_raw.shape[0], -1)
X_test  = (X_test_raw.astype(np.float32)  * 0.0001).reshape(X_test_raw.shape[0], -1)

num_classes = len(np.unique(y_train))
print(f"Train: {X_train.shape[0]} samples, {num_classes} classes")
print(f"Val:   {X_val.shape[0]} samples")
print(f"Test:  {X_test.shape[0]} samples")
print(f"y_train range: {y_train.min()} - {y_train.max()}")

# Class weights
y_train_np = np.asarray(y_train, dtype=np.int64)
classes, counts = np.unique(y_train_np, return_counts=True)
raw_weights = np.array([1.0 / counts[c] for c in range(num_classes)], dtype=np.float64)
sqrt_weights = np.sqrt(raw_weights)
sqrt_weights /= sqrt_weights.sum() / num_classes
class_weights_t = torch.tensor(sqrt_weights, dtype=torch.float32)

# Dataloaders
train_dataset = YourCustomDataset(X_train, y_train)
val_dataset   = YourCustomDataset(X_val,   y_val)
test_dataset  = YourCustomDataset(X_test,  y_test)

train_dataloader = DataLoader(train_dataset, batch_size=256, shuffle=True,
                               num_workers=4, pin_memory=False)
val_dataloader   = DataLoader(val_dataset,   batch_size=256, shuffle=False,
                               num_workers=4, pin_memory=False)
test_dataloader  = DataLoader(test_dataset,  batch_size=256, shuffle=False,
                               num_workers=4, pin_memory=False)

# Model
patch_size  = 3
in_channels = X_train.shape[1] // (patch_size**2)
max_epochs  = 200

model_l1 = ConvNet(
    lr=3e-4, num_classes=num_classes, in_channels=in_channels,
    patch_size=patch_size, class_weights=class_weights_t, max_epochs=max_epochs,
)

early_stop = pl.callbacks.EarlyStopping(monitor="val_loss", patience=20, mode="min")

trainer_l1 = pl.Trainer(
    accelerator="gpu", devices=1, max_epochs=max_epochs,
    gradient_clip_val=1.0, log_every_n_steps=50,
    enable_progress_bar=True, callbacks=[early_stop],
)

trainer_l1.fit(model_l1, train_dataloader, val_dataloader)
trainer_l1.test(model_l1, dataloaders=test_dataloader)

GPU available: True (cuda), used: True
TPU available: False, using: 0 TPU cores
💡 Tip: For seamless cloud logging and experiment tracking, try installing [litlogger](https://pypi.org/project/litlogger/) to enable LitLogger, which logs metrics and artifacts automatically to the Lightning Experiments platform.
💡 Tip: For seamless cloud uploads and versioning, try installing [litmodels](https://pypi.org/project/litmodels/) to enable LitModelCheckpoint, which syncs automatically with the Lightning model registry.
LOCAL_RANK: 0 - CUDA_VISIBLE_DEVICES: [0]


Train: 20000 samples, 5 classes
Val:   10000 samples
Test:  10000 samples
y_train range: 0 - 4


┏━━━┳━━━━━━━━━━━━┳━━━━━━━━━━━━┳━━━━━━━━┳━━━━━━━┳━━━━━━━┓
┃   ┃ Name       ┃ Type       ┃ Params ┃ Mode  ┃ FLOPs ┃
┡━━━╇━━━━━━━━━━━━╇━━━━━━━━━━━━╇━━━━━━━━╇━━━━━━━╇━━━━━━━┩
│ 0 │ features   │ Sequential │ 76.6 K │ train │     0 │
│ 1 │ classifier │ Sequential │  8.6 K │ train │     0 │
└───┴────────────┴────────────┴────────┴───────┴───────┘

Trainable params: 85.2 K                                                                                           
Non-trainable params: 0                                                                                            
Total params: 85.2 K                                                                                               
Total estimated model params size (MB): 0                                                                          
Modules in train mode: 14                                                                                          
Modules in eval mode: 0                                                                                            
Total FLOPs: 0

SLURM auto-requeueing enabled. Setting signal handlers.


Output()

LOCAL_RANK: 0 - CUDA_VISIBLE_DEVICES: [0]
SLURM auto-requeueing enabled. Setting signal handlers.


Output()

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━━━━┓
┃        Test metric        ┃       DataLoader 0        ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━━━━┩
│         test_acc          │    0.11990000307559967    │
│         test_loss         │     36.92877197265625     │
└───────────────────────────┴───────────────────────────┘

[{'test_loss': 36.92877197265625, 'test_acc': 0.11990000307559967}]

In [7]:
# Droppa class 3 (Wetlands, of fáar samples)
mask_tr = y_train != 3
mask_va = y_val   != 3
mask_te = y_test  != 3

X_train, y_train = X_train[mask_tr], y_train[mask_tr]
X_val,   y_val   = X_val[mask_va],   y_val[mask_va]
X_test,  y_test  = X_test[mask_te],  y_test[mask_te]

# Remap labels -- 4 verður 3
remap = {0: 0, 1: 1, 2: 2, 4: 3}
y_train = np.array([remap[l] for l in y_train], dtype=np.int64)
y_val   = np.array([remap[l] for l in y_val],   dtype=np.int64)
y_test  = np.array([remap[l] for l in y_test],  dtype=np.int64)

num_classes = 4
print(f"Train: {X_train.shape[0]}, classes: {np.unique(y_train)}")
print(f"Val:   {X_val.shape[0]}")
print(f"Test:  {X_test.shape[0]}")

Train: 19944, classes: [0 1 2 3]
Val:   9890
Test:  9913


In [8]:
# Recompute class weights
y_train_np = np.asarray(y_train, dtype=np.int64)
classes, counts = np.unique(y_train_np, return_counts=True)
raw_weights = np.array([1.0 / counts[c] for c in range(num_classes)], dtype=np.float64)
sqrt_weights = np.sqrt(raw_weights)
sqrt_weights /= sqrt_weights.sum() / num_classes
class_weights_t = torch.tensor(sqrt_weights, dtype=torch.float32)

# Datasets
train_dataset = YourCustomDataset(X_train, y_train)
val_dataset   = YourCustomDataset(X_val,   y_val)
test_dataset  = YourCustomDataset(X_test,  y_test)

train_dataloader = DataLoader(train_dataset, batch_size=256, shuffle=True,
                               num_workers=4, pin_memory=False)
val_dataloader   = DataLoader(val_dataset,   batch_size=256, shuffle=False,
                               num_workers=4, pin_memory=False)
test_dataloader  = DataLoader(test_dataset,  batch_size=256, shuffle=False,
                               num_workers=4, pin_memory=False)

model_l1 = ConvNet(
    lr=3e-4, num_classes=4, in_channels=in_channels,
    patch_size=patch_size, class_weights=class_weights_t, max_epochs=max_epochs,
)

early_stop = pl.callbacks.EarlyStopping(monitor="val_loss", patience=20, mode="min")

trainer_l1 = pl.Trainer(
    accelerator="gpu", devices=1, max_epochs=max_epochs,
    gradient_clip_val=1.0, log_every_n_steps=50,
    enable_progress_bar=True, callbacks=[early_stop],
)

trainer_l1.fit(model_l1, train_dataloader, val_dataloader)
trainer_l1.test(model_l1, dataloaders=test_dataloader)

GPU available: True (cuda), used: True
TPU available: False, using: 0 TPU cores
💡 Tip: For seamless cloud logging and experiment tracking, try installing [litlogger](https://pypi.org/project/litlogger/) to enable LitLogger, which logs metrics and artifacts automatically to the Lightning Experiments platform.
💡 Tip: For seamless cloud uploads and versioning, try installing [litmodels](https://pypi.org/project/litmodels/) to enable LitModelCheckpoint, which syncs automatically with the Lightning model registry.
LOCAL_RANK: 0 - CUDA_VISIBLE_DEVICES: [0]


┏━━━┳━━━━━━━━━━━━┳━━━━━━━━━━━━┳━━━━━━━━┳━━━━━━━┳━━━━━━━┓
┃   ┃ Name       ┃ Type       ┃ Params ┃ Mode  ┃ FLOPs ┃
┡━━━╇━━━━━━━━━━━━╇━━━━━━━━━━━━╇━━━━━━━━╇━━━━━━━╇━━━━━━━┩
│ 0 │ features   │ Sequential │ 76.6 K │ train │     0 │
│ 1 │ classifier │ Sequential │  8.5 K │ train │     0 │
└───┴────────────┴────────────┴────────┴───────┴───────┘

Trainable params: 85.1 K                                                                                           
Non-trainable params: 0                                                                                            
Total params: 85.1 K                                                                                               
Total estimated model params size (MB): 0                                                                          
Modules in train mode: 14                                                                                          
Modules in eval mode: 0                                                                                            
Total FLOPs: 0

SLURM auto-requeueing enabled. Setting signal handlers.


Output()

LOCAL_RANK: 0 - CUDA_VISIBLE_DEVICES: [0]
SLURM auto-requeueing enabled. Setting signal handlers.


Output()

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━━━━┓
┃        Test metric        ┃       DataLoader 0        ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━━━━┩
│         test_acc          │    0.12095228582620621    │
│         test_loss         │     72.6091079711914      │
└───────────────────────────┴───────────────────────────┘

[{'test_loss': 72.6091079711914, 'test_acc': 0.12095228582620621}]

GPU available: True (cuda), used: True
TPU available: False, using: 0 TPU cores
💡 Tip: For seamless cloud logging and experiment tracking, try installing [litlogger](https://pypi.org/project/litlogger/) to enable LitLogger, which logs metrics and artifacts automatically to the Lightning Experiments platform.
💡 Tip: For seamless cloud uploads and versioning, try installing [litmodels](https://pypi.org/project/litmodels/) to enable LitModelCheckpoint, which syncs automatically with the Lightning model registry.
LOCAL_RANK: 0 - CUDA_VISIBLE_DEVICES: [0]


Train: 99292 samples, 4 classes
Val:   29740 samples
Test:  29930 samples
y_train range: 0 - 3
  Artificial: 7400 (7.5%)
  Agricultural: 56916 (57.3%)
  Forest: 28386 (28.6%)
  Water: 6590 (6.6%)

Class weights: tensor([1.3648, 0.4921, 0.6968, 1.4462])


┏━━━┳━━━━━━━━━━━━┳━━━━━━━━━━━━┳━━━━━━━━┳━━━━━━━┳━━━━━━━┓
┃   ┃ Name       ┃ Type       ┃ Params ┃ Mode  ┃ FLOPs ┃
┡━━━╇━━━━━━━━━━━━╇━━━━━━━━━━━━╇━━━━━━━━╇━━━━━━━╇━━━━━━━┩
│ 0 │ features   │ Sequential │ 76.6 K │ train │     0 │
│ 1 │ classifier │ Sequential │  8.5 K │ train │     0 │
└───┴────────────┴────────────┴────────┴───────┴───────┘

Trainable params: 85.1 K                                                                                           
Non-trainable params: 0                                                                                            
Total params: 85.1 K                                                                                               
Total estimated model params size (MB): 0                                                                          
Modules in train mode: 14                                                                                          
Modules in eval mode: 0                                                                                            
Total FLOPs: 0

SLURM auto-requeueing enabled. Setting signal handlers.


Output()

LOCAL_RANK: 0 - CUDA_VISIBLE_DEVICES: [0]
SLURM auto-requeueing enabled. Setting signal handlers.


Output()

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━━━━┓
┃        Test metric        ┃       DataLoader 0        ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━━━━┩
│         test_acc          │    0.19321750104427338    │
│         test_loss         │    20.001558303833008     │
└───────────────────────────┴───────────────────────────┘


========== RESULTS ==========
Overall accuracy:  0.1932
Balanced accuracy: 0.0483
Macro-F1:          0.0810

Predicted classes: (array([0, 1, 2, 3]), array([ 6252, 16691,  5783,  1204]))

Confusion matrix:
[[    0     0     0     0]
 [    0     0     0     0]
 [ 6252 16691  5783  1204]
 [    0     0     0     0]]


In [13]:
print("y_test unique:", np.unique(y_test, return_counts=True))
print("y_true unique:", np.unique(y_true, return_counts=True))
print("y_pred unique:", np.unique(y_pred, return_counts=True))

y_test unique: (array([0, 1, 2, 3]), array([ 6252, 16691,  5783,  1204]))
y_true unique: (array([2]), array([29930]))
y_pred unique: (array([0, 1, 2, 3]), array([ 6252, 16691,  5783,  1204]))


In [14]:
xb, yb = next(iter(test_dataloader))
print("Batch labels:", yb[:20])
print("Unique in batch:", torch.unique(yb))

Batch labels: tensor([1, 1, 2, 1, 1, 2, 1, 1, 0, 1, 0, 1, 3, 1, 1, 0, 0, 1, 1, 0])
Unique in batch: tensor([0, 1, 2, 3])


In [16]:
model_geo.eval()
preds_all, targets_all = [], []
with torch.no_grad():
    for xb, yb in test_dataloader:
        logits = model_geo(xb.to('cuda'))
        preds_all.append(logits.argmax(1).cpu().numpy())
        targets_all.append(yb.numpy())

y_true2 = np.concatenate(targets_all).astype(np.int64)
y_pred2 = np.concatenate(preds_all).astype(np.int64)

print("y_true unique:", np.unique(y_true2, return_counts=True))
print("y_pred unique:", np.unique(y_pred2, return_counts=True))
oa, ba, mf1, cm = _imbalanced_metrics(y_true2, y_pred2, num_classes)
print(f"\nOverall accuracy:  {oa:.4f}")
print(f"Balanced accuracy: {ba:.4f}")
print(f"Macro-F1:          {mf1:.4f}")
print(f"\nConfusion matrix:\n{cm}")

y_true unique: (array([0, 1, 2, 3]), array([ 6252, 16691,  5783,  1204]))
y_pred unique: (array([2]), array([29930]))

Overall accuracy:  0.1932
Balanced accuracy: 0.2500
Macro-F1:          0.0810

Confusion matrix:
[[    0     0  6252     0]
 [    0     0 16691     0]
 [    0     0  5783     0]
 [    0     0  1204     0]]


## CNN with Column Split, Top 4 Classes

Trains CNN on vertical column split using the 4 most frequent CORINE classes.
Achieves only 14.5% overall accuracy -- the model predicts only Broad-leaved
forest (majority class in test region). This confirms that the column split
produces significant class distribution mismatch between train and test regions,
motivating the switch to block cross-validation.

In [19]:
import os
os.environ["CUDA_LAUNCH_BLOCKING"] = "1"
os.environ["CUDA_VISIBLE_DEVICES"] = "0"

import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
import lightning.pytorch as pl
from torch.utils.data import DataLoader, Dataset
from pathlib import Path

# ============================================================
# Dataset og Model
# ============================================================
class YourCustomDataset(Dataset):
    def __init__(self, data, labels):
        self.data = data
        self.labels = labels
    def __len__(self):
        return len(self.data)
    def __getitem__(self, idx):
        return self.data[idx], self.labels[idx]

class ConvNet(pl.LightningModule):
    def __init__(self, num_classes=4, in_channels=4, patch_size=3,
                 lr=3e-4, max_epochs=200, class_weights=None):
        super().__init__()
        self.lr = lr
        self.in_channels = in_channels
        self.patch_size = patch_size
        self.num_classes = num_classes
        self.max_epochs = max_epochs
        self.features = nn.Sequential(
            nn.Conv2d(in_channels, 64, kernel_size=3, padding=1),
            nn.BatchNorm2d(64), nn.ReLU(inplace=True),
            nn.Conv2d(64, 128, kernel_size=3, padding=1),
            nn.BatchNorm2d(128), nn.ReLU(inplace=True),
            nn.AdaptiveAvgPool2d((1, 1)),
        )
        self.classifier = nn.Sequential(
            nn.Flatten(), nn.Linear(128, 64),
            nn.ReLU(), nn.Dropout(0.3),
            nn.Linear(64, num_classes),
        )
        if class_weights is not None:
            self.register_buffer("class_weights", class_weights.float())
        else:
            self.class_weights = None

    def _to_image(self, x):
        x = x.float()
        h = w = self.patch_size
        c = self.in_channels
        if x.dim() == 2:
            x = x.view(x.size(0), h, w, c).permute(0, 3, 1, 2).contiguous()
        return x

    def forward(self, x):
        return self.classifier(self.features(self._to_image(x)))

    def _loss(self, logits, y):
        return F.cross_entropy(logits, y, weight=self.class_weights)

    def training_step(self, batch, _):
        x, y = batch
        loss = self._loss(self(x), y.long())
        self.log("train_loss", loss, prog_bar=True)
        return loss

    def validation_step(self, batch, _):
        x, y = batch
        logits = self(x)
        loss = self._loss(logits, y.long())
        acc = (logits.argmax(1) == y).float().mean()
        self.log("val_loss", loss, prog_bar=True)
        self.log("val_acc", acc, prog_bar=True)

    def test_step(self, batch, _):
        x, y = batch
        logits = self(x)
        loss = self._loss(logits, y.long())
        acc = (logits.argmax(1) == y).float().mean()
        self.log("test_loss", loss)
        self.log("test_acc", acc, prog_bar=True)

    def configure_optimizers(self):
        opt = torch.optim.AdamW(self.parameters(), lr=self.lr, weight_decay=1e-4)
        sched = torch.optim.lr_scheduler.CosineAnnealingLR(opt, T_max=self.max_epochs)
        return {"optimizer": opt, "lr_scheduler": sched}

# ============================================================
# Load data
# ============================================================
FINAL_DIR = Path("/p/scratch/training2600/TeamGudnason/training_data/final_geo_col")

X_train_raw = np.load(FINAL_DIR / "patches_train.npz")['patches'][:, :, :, :4]
y_train_raw = np.load(FINAL_DIR / "labels_train.npy")
X_val_raw   = np.load(FINAL_DIR / "patches_val.npz")['patches'][:, :, :, :4]
y_val_raw   = np.load(FINAL_DIR / "labels_val.npy")
X_test_raw  = np.load(FINAL_DIR / "patches_test.npz")['patches'][:, :, :, :4]
y_test_raw  = np.load(FINAL_DIR / "labels_test.npy")

# Normalize
X_train_all = (X_train_raw.astype(np.float32) * 0.0001).reshape(X_train_raw.shape[0], -1)
X_val_all   = (X_val_raw.astype(np.float32)   * 0.0001).reshape(X_val_raw.shape[0], -1)
X_test_all  = (X_test_raw.astype(np.float32)  * 0.0001).reshape(X_test_raw.shape[0], -1)

# Remap all labels first
unique_labels = np.unique(y_train_raw)
label_to_idx  = {int(lbl): i for i, lbl in enumerate(unique_labels)}
idx_to_label  = {i: int(lbl) for i, lbl in enumerate(unique_labels)}

y_train_all = np.array([label_to_idx[int(l)] for l in y_train_raw], dtype=np.int64)
y_val_all   = np.array([label_to_idx.get(int(l), -1) for l in y_val_raw],   dtype=np.int64)
y_test_all  = np.array([label_to_idx.get(int(l), -1) for l in y_test_raw],  dtype=np.int64)

X_val_all,  y_val_all  = X_val_all[y_val_all != -1],   y_val_all[y_val_all != -1]
X_test_all, y_test_all = X_test_all[y_test_all != -1], y_test_all[y_test_all != -1]

# Keep only top 4 classes by frequency
# CORINE 12 (idx 3), CORINE 23 (idx 9), CORINE 41 (idx 16), CORINE 18 (idx 6)
KEEP = [3, 9, 16, 6]
CLASS_NAMES = {3: "Arable land", 9: "Broad-leaved forest", 16: "Water bodies", 6: "Pastures"}

mask_tr = np.isin(y_train_all, KEEP)
mask_va = np.isin(y_val_all,   KEEP)
mask_te = np.isin(y_test_all,  KEEP)

X_train, y_tr = X_train_all[mask_tr], y_train_all[mask_tr]
X_val,   y_va = X_val_all[mask_va],   y_val_all[mask_va]
X_test,  y_te = X_test_all[mask_te],  y_test_all[mask_te]

# Remap to 0..3
remap = {3: 0, 9: 1, 16: 2, 6: 3}
y_train = np.array([remap[l] for l in y_tr], dtype=np.int64)
y_val   = np.array([remap[l] for l in y_va], dtype=np.int64)
y_test  = np.array([remap[l] for l in y_te], dtype=np.int64)

num_classes = 4
FINAL_NAMES = {0: "Arable land", 1: "Broad-leaved forest", 2: "Water bodies", 3: "Pastures"}

print(f"Train: {X_train.shape[0]} samples, {num_classes} classes")
print(f"Val:   {X_val.shape[0]} samples")
print(f"Test:  {X_test.shape[0]} samples")
print(f"y_train range: {y_train.min()} - {y_train.max()}")

classes, counts = np.unique(y_train, return_counts=True)
for c, n in zip(classes, counts):
    print(f"  Class {c} ({FINAL_NAMES[c]}): {n} ({100*n/len(y_train):.1f}%)")

# ============================================================
# Class weights
# ============================================================
raw_weights = np.zeros(num_classes, dtype=np.float64)
for c, n in zip(classes, counts):
    raw_weights[int(c)] = 1.0 / n
sqrt_weights = np.sqrt(raw_weights)
sqrt_weights /= sqrt_weights.sum() / num_classes
class_weights_t = torch.tensor(sqrt_weights, dtype=torch.float32)
print(f"\nClass weights: {class_weights_t}")

# ============================================================
# Dataloaders
# ============================================================
train_dataset = YourCustomDataset(X_train, y_train)
val_dataset   = YourCustomDataset(X_val,   y_val)
test_dataset  = YourCustomDataset(X_test,  y_test)

train_dataloader = DataLoader(train_dataset, batch_size=512, shuffle=True,
                               num_workers=4, pin_memory=False)
val_dataloader   = DataLoader(val_dataset,   batch_size=512, shuffle=False,
                               num_workers=4, pin_memory=False)
test_dataloader  = DataLoader(test_dataset,  batch_size=512, shuffle=False,
                               num_workers=4, pin_memory=False)

# ============================================================
# Model + Trainer
# ============================================================
patch_size  = 3
in_channels = X_train.shape[1] // (patch_size**2)
max_epochs  = 200

model_geo = ConvNet(
    lr=3e-4, num_classes=num_classes, in_channels=in_channels,
    patch_size=patch_size, class_weights=class_weights_t, max_epochs=max_epochs,
)

early_stop = pl.callbacks.EarlyStopping(monitor="val_loss", patience=20, mode="min")

trainer_geo = pl.Trainer(
    accelerator="gpu", devices=1, max_epochs=max_epochs,
    gradient_clip_val=1.0, log_every_n_steps=50,
    enable_progress_bar=True, callbacks=[early_stop],
)

trainer_geo.fit(model_geo, train_dataloader, val_dataloader)
trainer_geo.test(model_geo, dataloaders=test_dataloader)

# ============================================================
# Diagnostics
# ============================================================
model_geo.eval()
model_geo = model_geo.cuda()
preds_all, targets_all = [], []
with torch.no_grad():
    for xb, yb in test_dataloader:
        logits = model_geo(xb.to('cuda'))
        preds_all.append(logits.argmax(1).cpu().numpy())
        targets_all.append(yb.numpy())

y_true = np.concatenate(targets_all).astype(np.int64)
y_pred = np.concatenate(preds_all).astype(np.int64)

cm = np.zeros((num_classes, num_classes), dtype=np.int64)
for t, p in zip(y_true, y_pred):
    cm[t, p] += 1

tp        = np.diag(cm).astype(np.float64)
support   = cm.sum(axis=1).astype(np.float64)
pred_c    = cm.sum(axis=0).astype(np.float64)
recall    = np.divide(tp, support, out=np.zeros_like(tp), where=support > 0)
precision = np.divide(tp, pred_c,  out=np.zeros_like(tp), where=pred_c > 0)
f1        = np.divide(2*precision*recall, precision+recall,
                      out=np.zeros_like(tp), where=(precision+recall) > 0)

oa  = tp.sum() / cm.sum()
ba  = recall.mean()
mf1 = f1.mean()

print(f"\n========== RESULTS ==========")
print(f"Overall accuracy:  {oa:.4f}")
print(f"Balanced accuracy: {ba:.4f}")
print(f"Macro-F1:          {mf1:.4f}")
print(f"\nPer-class metrics:")
print(f"{'Class':<25} {'Support':<10} {'Recall':<10} {'Precision':<12} {'F1':<10}")
for i in range(num_classes):
    print(f"{FINAL_NAMES[i]:<25} {int(support[i]):<10} {recall[i]:<10.3f} {precision[i]:<12.3f} {f1[i]:<10.3f}")

print(f"\nConfusion matrix:")
print(f"{'':25}", end="")
for i in range(num_classes):
    print(f"{FINAL_NAMES[i][:10]:<12}", end="")
print()
for i in range(num_classes):
    print(f"{FINAL_NAMES[i]:<25}", end="")
    for j in range(num_classes):
        print(f"{cm[i,j]:<12}", end="")
    print()

GPU available: True (cuda), used: True
TPU available: False, using: 0 TPU cores
💡 Tip: For seamless cloud logging and experiment tracking, try installing [litlogger](https://pypi.org/project/litlogger/) to enable LitLogger, which logs metrics and artifacts automatically to the Lightning Experiments platform.
💡 Tip: For seamless cloud uploads and versioning, try installing [litmodels](https://pypi.org/project/litmodels/) to enable LitModelCheckpoint, which syncs automatically with the Lightning model registry.
LOCAL_RANK: 0 - CUDA_VISIBLE_DEVICES: [0]


Train: 21284 samples, 4 classes
Val:   4492 samples
Test:  5292 samples
y_train range: 0 - 3
  Class 0 (Arable land): 4 (0.0%)
  Class 1 (Broad-leaved forest): 992 (4.7%)
  Class 2 (Water bodies): 20197 (94.9%)
  Class 3 (Pastures): 91 (0.4%)

Class weights: tensor([3.1074, 0.1973, 0.0437, 0.6515])


┏━━━┳━━━━━━━━━━━━┳━━━━━━━━━━━━┳━━━━━━━━┳━━━━━━━┳━━━━━━━┓
┃   ┃ Name       ┃ Type       ┃ Params ┃ Mode  ┃ FLOPs ┃
┡━━━╇━━━━━━━━━━━━╇━━━━━━━━━━━━╇━━━━━━━━╇━━━━━━━╇━━━━━━━┩
│ 0 │ features   │ Sequential │ 76.6 K │ train │     0 │
│ 1 │ classifier │ Sequential │  8.5 K │ train │     0 │
└───┴────────────┴────────────┴────────┴───────┴───────┘

Trainable params: 85.1 K                                                                                           
Non-trainable params: 0                                                                                            
Total params: 85.1 K                                                                                               
Total estimated model params size (MB): 0                                                                          
Modules in train mode: 14                                                                                          
Modules in eval mode: 0                                                                                            
Total FLOPs: 0

SLURM auto-requeueing enabled. Setting signal handlers.


...

/p/project1/training2600/TeamGudnason/envs/danube/lib/python3.12/site-packages/lightning/pytorch/loops/fit_loop.py:
317: The number of training batches (42) is smaller than the logging interval Trainer(log_every_n_steps=50). Set a 
lower value for log_every_n_steps if you want to see logs for the training epoch.

LOCAL_RANK: 0 - CUDA_VISIBLE_DEVICES: [0]
SLURM auto-requeueing enabled. Setting signal handlers.


Output()

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━━━━┓
┃        Test metric        ┃       DataLoader 0        ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━━━━┩
│         test_acc          │    0.1447467803955078     │
│         test_loss         │    30.471006393432617     │
└───────────────────────────┴───────────────────────────┘


========== RESULTS ==========
Overall accuracy:  0.1447
Balanced accuracy: 0.2500
Macro-F1:          0.0632

Per-class metrics:
Class                     Support    Recall     Precision    F1        
Arable land               33         0.000      0.000        0.000     
Broad-leaved forest       766        1.000      0.145        0.253     
Water bodies              4470       0.000      0.000        0.000     
Pastures                  23         0.000      0.000        0.000     

Confusion matrix:
                         Arable lan  Broad-leav  Water bodi  Pastures    
Arable land              0           33          0           0           
Broad-leaved forest      0           766         0           0           
Water bodies             0           4470        0           0           
Pastures                 0           23          0           0           


In [20]:
# Skoðum hvað KEEP_CLASSES eru í raun
print("y_train unique before filter:", np.unique(y_train, return_counts=True))

y_train unique before filter: (array([0, 1, 2, 3]), array([    4,   992, 20197,    91]))


## CNN with Column Split, Top 4 Classes (Improved Label Mapping)

Trains CNN on vertical column split with automatic top-4 class selection.
Achieves 67.6% overall accuracy but only 25% balanced accuracy -- the model
predicts only CORINE 12 (arable land). This reveals the fundamental problem
with column splits: class distribution differs significantly between the left
(train) and right (test) portions of the tile, making fair evaluation impossible.
This motivated the switch to block cross-validation.

In [1]:
import os
os.environ["CUDA_LAUNCH_BLOCKING"] = "1"
os.environ["CUDA_VISIBLE_DEVICES"] = "0"

import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
import lightning.pytorch as pl
from torch.utils.data import DataLoader, Dataset
from pathlib import Path

# ============================================================
# Dataset og Model
# ============================================================
class YourCustomDataset(Dataset):
    def __init__(self, data, labels):
        self.data = data
        self.labels = labels
    def __len__(self):
        return len(self.data)
    def __getitem__(self, idx):
        return self.data[idx], self.labels[idx]

class ConvNet(pl.LightningModule):
    def __init__(self, num_classes=4, in_channels=4, patch_size=3,
                 lr=3e-4, max_epochs=200, class_weights=None):
        super().__init__()
        self.lr = lr
        self.in_channels = in_channels
        self.patch_size = patch_size
        self.num_classes = num_classes
        self.max_epochs = max_epochs
        self.features = nn.Sequential(
            nn.Conv2d(in_channels, 64, kernel_size=3, padding=1),
            nn.BatchNorm2d(64), nn.ReLU(inplace=True),
            nn.Conv2d(64, 128, kernel_size=3, padding=1),
            nn.BatchNorm2d(128), nn.ReLU(inplace=True),
            nn.AdaptiveAvgPool2d((1, 1)),
        )
        self.classifier = nn.Sequential(
            nn.Flatten(), nn.Linear(128, 64),
            nn.ReLU(), nn.Dropout(0.3),
            nn.Linear(64, num_classes),
        )
        if class_weights is not None:
            self.register_buffer("class_weights", class_weights.float())
        else:
            self.class_weights = None

    def _to_image(self, x):
        x = x.float()
        h = w = self.patch_size
        c = self.in_channels
        if x.dim() == 2:
            x = x.view(x.size(0), h, w, c).permute(0, 3, 1, 2).contiguous()
        return x

    def forward(self, x):
        return self.classifier(self.features(self._to_image(x)))

    def _loss(self, logits, y):
        return F.cross_entropy(logits, y, weight=self.class_weights)

    def training_step(self, batch, _):
        x, y = batch
        loss = self._loss(self(x), y.long())
        self.log("train_loss", loss, prog_bar=True)
        return loss

    def validation_step(self, batch, _):
        x, y = batch
        logits = self(x)
        loss = self._loss(logits, y.long())
        acc = (logits.argmax(1) == y).float().mean()
        self.log("val_loss", loss, prog_bar=True)
        self.log("val_acc", acc, prog_bar=True)

    def test_step(self, batch, _):
        x, y = batch
        logits = self(x)
        loss = self._loss(logits, y.long())
        acc = (logits.argmax(1) == y).float().mean()
        self.log("test_loss", loss)
        self.log("test_acc", acc, prog_bar=True)

    def configure_optimizers(self):
        opt = torch.optim.AdamW(self.parameters(), lr=self.lr, weight_decay=1e-4)
        sched = torch.optim.lr_scheduler.CosineAnnealingLR(opt, T_max=self.max_epochs)
        return {"optimizer": opt, "lr_scheduler": sched}

# ============================================================
# Load data
# ============================================================
FINAL_DIR = Path("/p/scratch/training2600/TeamGudnason/training_data/final_geo_col")

X_train_raw = np.load(FINAL_DIR / "patches_train.npz")['patches'][:, :, :, :4]
y_train_raw = np.load(FINAL_DIR / "labels_train.npy")
X_val_raw   = np.load(FINAL_DIR / "patches_val.npz")['patches'][:, :, :, :4]
y_val_raw   = np.load(FINAL_DIR / "labels_val.npy")
X_test_raw  = np.load(FINAL_DIR / "patches_test.npz")['patches'][:, :, :, :4]
y_test_raw  = np.load(FINAL_DIR / "labels_test.npy")

# Normalize
X_all_tr = (X_train_raw.astype(np.float32) * 0.0001).reshape(X_train_raw.shape[0], -1)
X_all_va = (X_val_raw.astype(np.float32)   * 0.0001).reshape(X_val_raw.shape[0], -1)
X_all_te = (X_test_raw.astype(np.float32)  * 0.0001).reshape(X_test_raw.shape[0], -1)

# Remap ALL labels first
unique_all = np.unique(y_train_raw)
label_to_idx = {int(lbl): i for i, lbl in enumerate(unique_all)}
idx_to_corine = {i: int(lbl) for i, lbl in enumerate(unique_all)}

y_all_tr = np.array([label_to_idx[int(l)] for l in y_train_raw], dtype=np.int64)
y_all_va = np.array([label_to_idx.get(int(l), -1) for l in y_val_raw], dtype=np.int64)
y_all_te = np.array([label_to_idx.get(int(l), -1) for l in y_test_raw], dtype=np.int64)

# Remove unseen
X_all_va, y_all_va = X_all_va[y_all_va != -1], y_all_va[y_all_va != -1]
X_all_te, y_all_te = X_all_te[y_all_te != -1], y_all_te[y_all_te != -1]

# Drop classes < 200 samples
classes_full, counts_full = np.unique(y_all_tr, return_counts=True)
valid = set(classes_full[counts_full >= 200])
mask_tr = np.isin(y_all_tr, list(valid))
mask_va = np.isin(y_all_va, list(valid))
mask_te = np.isin(y_all_te, list(valid))
X_all_tr, y_all_tr = X_all_tr[mask_tr], y_all_tr[mask_tr]
X_all_va, y_all_va = X_all_va[mask_va], y_all_va[mask_va]
X_all_te, y_all_te = X_all_te[mask_te], y_all_te[mask_te]

# Final contiguous remap
unique_final = np.unique(y_all_tr)
final_map = {int(l): i for i, l in enumerate(unique_final)}
y_all_tr = np.array([final_map[l] for l in y_all_tr], dtype=np.int64)
y_all_va = np.array([final_map[l] for l in y_all_va], dtype=np.int64)
y_all_te = np.array([final_map[l] for l in y_all_te], dtype=np.int64)

print("Full dataset (17 classes):")
classes_17, counts_17 = np.unique(y_all_tr, return_counts=True)
for c, n in zip(classes_17, counts_17):
    corine = idx_to_corine[unique_final[int(c)]]
    print(f"  idx {c} (CORINE {corine}): {n} ({100*n/len(y_all_tr):.1f}%)")

# ============================================================
# Keep top 4 classes by count
# ============================================================
top4_idx = classes_17[np.argsort(counts_17)[::-1][:4]]
print(f"\nTop 4 class indices: {top4_idx}")
for i in top4_idx:
    print(f"  idx {i} (CORINE {idx_to_corine[unique_final[int(i)]]}): {counts_17[i]}")

mask_tr = np.isin(y_all_tr, top4_idx)
mask_va = np.isin(y_all_va, top4_idx)
mask_te = np.isin(y_all_te, top4_idx)

X_train, y_train_raw2 = X_all_tr[mask_tr], y_all_tr[mask_tr]
X_val,   y_val_raw2   = X_all_va[mask_va], y_all_va[mask_va]
X_test,  y_test_raw2  = X_all_te[mask_te], y_all_te[mask_te]

# Remap to 0..3
top4_map = {int(c): i for i, c in enumerate(sorted(top4_idx))}
y_train = np.array([top4_map[l] for l in y_train_raw2], dtype=np.int64)
y_val   = np.array([top4_map[l] for l in y_val_raw2],   dtype=np.int64)
y_test  = np.array([top4_map[l] for l in y_test_raw2],  dtype=np.int64)

num_classes = 4
classes_k, counts_k = np.unique(y_train, return_counts=True)
CLASS_NAMES = {i: f"CORINE {idx_to_corine[unique_final[int(sorted(top4_idx)[i])]]}"
               for i in range(4)}

print(f"\nFiltered dataset (4 classes):")
print(f"Train: {X_train.shape[0]}")
print(f"Val:   {X_val.shape[0]}")
print(f"Test:  {X_test.shape[0]}")
for c, n in zip(classes_k, counts_k):
    print(f"  {CLASS_NAMES[c]}: {n} ({100*n/len(y_train):.1f}%)")

# ============================================================
# Class weights
# ============================================================
raw_weights = np.array([1.0/counts_k[c] for c in range(num_classes)], dtype=np.float64)
sqrt_weights = np.sqrt(raw_weights)
sqrt_weights /= sqrt_weights.sum() / num_classes
class_weights_t = torch.tensor(sqrt_weights, dtype=torch.float32)
print(f"\nClass weights: {class_weights_t}")

# ============================================================
# Dataloaders
# ============================================================
train_dataset = YourCustomDataset(X_train, y_train)
val_dataset   = YourCustomDataset(X_val,   y_val)
test_dataset  = YourCustomDataset(X_test,  y_test)

train_dataloader = DataLoader(train_dataset, batch_size=512, shuffle=True,
                               num_workers=4, pin_memory=False)
val_dataloader   = DataLoader(val_dataset,   batch_size=512, shuffle=False,
                               num_workers=4, pin_memory=False)
test_dataloader  = DataLoader(test_dataset,  batch_size=512, shuffle=False,
                               num_workers=4, pin_memory=False)

# ============================================================
# Model + Trainer
# ============================================================
patch_size  = 3
in_channels = X_train.shape[1] // (patch_size**2)
max_epochs  = 200

model_top4 = ConvNet(
    lr=3e-4, num_classes=4, in_channels=in_channels,
    patch_size=patch_size, class_weights=class_weights_t, max_epochs=max_epochs,
)

early_stop = pl.callbacks.EarlyStopping(monitor="val_loss", patience=20, mode="min")

trainer_top4 = pl.Trainer(
    accelerator="gpu", devices=1, max_epochs=max_epochs,
    gradient_clip_val=1.0, log_every_n_steps=10,
    enable_progress_bar=True, callbacks=[early_stop],
)

trainer_top4.fit(model_top4, train_dataloader, val_dataloader)
trainer_top4.test(model_top4, dataloaders=test_dataloader)

# ============================================================
# Diagnostics
# ============================================================
model_top4.eval()
model_top4 = model_top4.cuda()
preds_all, targets_all = [], []
with torch.no_grad():
    for xb, yb in test_dataloader:
        logits = model_top4(xb.cuda())
        preds_all.append(logits.argmax(1).cpu().numpy())
        targets_all.append(yb.numpy())

y_true = np.concatenate(targets_all).astype(np.int64)
y_pred = np.concatenate(preds_all).astype(np.int64)

cm = np.zeros((4, 4), dtype=np.int64)
for t, p in zip(y_true, y_pred):
    cm[t, p] += 1

tp        = np.diag(cm).astype(np.float64)
support   = cm.sum(axis=1).astype(np.float64)
pred_c    = cm.sum(axis=0).astype(np.float64)
recall    = np.divide(tp, support, out=np.zeros_like(tp), where=support > 0)
precision = np.divide(tp, pred_c,  out=np.zeros_like(tp), where=pred_c > 0)
f1        = np.divide(2*precision*recall, precision+recall,
                      out=np.zeros_like(tp), where=(precision+recall) > 0)

print(f"\n========== RESULTS ==========")
print(f"Overall accuracy:  {tp.sum()/cm.sum():.4f}")
print(f"Balanced accuracy: {recall.mean():.4f}")
print(f"Macro-F1:          {f1.mean():.4f}")
print(f"\nPer-class:")
for i in range(4):
    print(f"  {CLASS_NAMES[i]:<20} recall={recall[i]:.3f} f1={f1[i]:.3f} support={int(support[i])}")
print(f"\nConfusion matrix:\n{cm}")
print(f"\nPredicted distribution: {np.unique(y_pred, return_counts=True)}")

GPU available: True (cuda), used: True
TPU available: False, using: 0 TPU cores
💡 Tip: For seamless cloud logging and experiment tracking, try installing [litlogger](https://pypi.org/project/litlogger/) to enable LitLogger, which logs metrics and artifacts automatically to the Lightning Experiments platform.
💡 Tip: For seamless cloud uploads and versioning, try installing [litmodels](https://pypi.org/project/litmodels/) to enable LitModelCheckpoint, which syncs automatically with the Lightning model registry.


Full dataset (17 classes):
  idx 0 (CORINE 2): 4798 (4.8%)
  idx 1 (CORINE 3): 1100 (1.1%)
  idx 2 (CORINE 11): 992 (1.0%)
  idx 3 (CORINE 12): 44778 (45.0%)
  idx 4 (CORINE 15): 1403 (1.4%)
  idx 5 (CORINE 16): 462 (0.5%)
  idx 6 (CORINE 18): 5456 (5.5%)
  idx 7 (CORINE 20): 2796 (2.8%)
  idx 8 (CORINE 21): 2021 (2.0%)
  idx 9 (CORINE 23): 20197 (20.3%)
  idx 10 (CORINE 24): 980 (1.0%)
  idx 11 (CORINE 25): 1351 (1.4%)
  idx 12 (CORINE 26): 2484 (2.5%)
  idx 13 (CORINE 29): 3374 (3.4%)
  idx 14 (CORINE 35): 616 (0.6%)
  idx 15 (CORINE 40): 515 (0.5%)
  idx 16 (CORINE 41): 6075 (6.1%)

Top 4 class indices: [ 3  9 16  6]
  idx 3 (CORINE 12): 44778
  idx 9 (CORINE 23): 20197
  idx 16 (CORINE 41): 6075
  idx 6 (CORINE 18): 5456

Filtered dataset (4 classes):
Train: 76506
Val:   23385
Test:  19826
  CORINE 12: 44778 (58.5%)
  CORINE 18: 5456 (7.1%)
  CORINE 23: 20197 (26.4%)
  CORINE 41: 6075 (7.9%)

Class weights: tensor([0.4957, 1.4202, 0.7381, 1.3459])


You are using a CUDA device ('NVIDIA A100-SXM4-40GB') that has Tensor Cores. To properly utilize them, you should set `torch.set_float32_matmul_precision('medium' | 'high')` which will trade-off precision for performance. For more details, read https://pytorch.org/docs/stable/generated/torch.set_float32_matmul_precision.html#torch.set_float32_matmul_precision
LOCAL_RANK: 0 - CUDA_VISIBLE_DEVICES: [0]


┏━━━┳━━━━━━━━━━━━┳━━━━━━━━━━━━┳━━━━━━━━┳━━━━━━━┳━━━━━━━┓
┃   ┃ Name       ┃ Type       ┃ Params ┃ Mode  ┃ FLOPs ┃
┡━━━╇━━━━━━━━━━━━╇━━━━━━━━━━━━╇━━━━━━━━╇━━━━━━━╇━━━━━━━┩
│ 0 │ features   │ Sequential │ 76.6 K │ train │     0 │
│ 1 │ classifier │ Sequential │  8.5 K │ train │     0 │
└───┴────────────┴────────────┴────────┴───────┴───────┘

Trainable params: 85.1 K                                                                                           
Non-trainable params: 0                                                                                            
Total params: 85.1 K                                                                                               
Total estimated model params size (MB): 0                                                                          
Modules in train mode: 14                                                                                          
Modules in eval mode: 0                                                                                            
Total FLOPs: 0

...

/p/project1/training2600/TeamGudnason/envs/danube/lib/python3.12/site-packages/lightning/pytorch/utilities/_pytree.
py:21: `isinstance(treespec, LeafSpec)` is deprecated, use `isinstance(treespec, TreeSpec) and treespec.is_leaf()` 
instead.

LOCAL_RANK: 0 - CUDA_VISIBLE_DEVICES: [0]
SLURM auto-requeueing enabled. Setting signal handlers.


Output()

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━━━━┓
┃        Test metric        ┃       DataLoader 0        ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━━━━┩
│         test_acc          │    0.6761828064918518     │
│         test_loss         │     24.81985855102539     │
└───────────────────────────┴───────────────────────────┘


========== RESULTS ==========
Overall accuracy:  0.6762
Balanced accuracy: 0.2500
Macro-F1:          0.2017

Per-class:
  CORINE 12            recall=1.000 f1=0.807 support=13406
  CORINE 18            recall=0.000 f1=0.000 support=1648
  CORINE 23            recall=0.000 f1=0.000 support=4470
  CORINE 41            recall=0.000 f1=0.000 support=302

Confusion matrix:
[[13406     0     0     0]
 [ 1648     0     0     0]
 [ 4470     0     0     0]
 [  302     0     0     0]]

Predicted distribution: (array([0]), array([19826]))


## CNN with Column Split, 16 Bands (Time Series)

Tests whether using all 16 spectral bands (4 acquisitions × 4 bands) improves
column split performance. Achieves the same 67.6% overall / 25% balanced accuracy
as the single-date version, confirming that temporal information alone cannot
overcome the class distribution mismatch between left and right tile regions.

In [2]:
import os
os.environ["CUDA_LAUNCH_BLOCKING"] = "1"
os.environ["CUDA_VISIBLE_DEVICES"] = "0"

import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
import lightning.pytorch as pl
from torch.utils.data import DataLoader, Dataset
from pathlib import Path

class YourCustomDataset(Dataset):
    def __init__(self, data, labels):
        self.data = data
        self.labels = labels
    def __len__(self):
        return len(self.data)
    def __getitem__(self, idx):
        return self.data[idx], self.labels[idx]

class ConvNet(pl.LightningModule):
    def __init__(self, num_classes=4, in_channels=16, patch_size=3,
                 lr=3e-4, max_epochs=200, class_weights=None):
        super().__init__()
        self.lr = lr
        self.in_channels = in_channels
        self.patch_size = patch_size
        self.num_classes = num_classes
        self.max_epochs = max_epochs
        self.features = nn.Sequential(
            nn.Conv2d(in_channels, 64, kernel_size=3, padding=1),
            nn.BatchNorm2d(64), nn.ReLU(inplace=True),
            nn.Conv2d(64, 128, kernel_size=3, padding=1),
            nn.BatchNorm2d(128), nn.ReLU(inplace=True),
            nn.AdaptiveAvgPool2d((1, 1)),
        )
        self.classifier = nn.Sequential(
            nn.Flatten(), nn.Linear(128, 64),
            nn.ReLU(), nn.Dropout(0.3),
            nn.Linear(64, num_classes),
        )
        if class_weights is not None:
            self.register_buffer("class_weights", class_weights.float())
        else:
            self.class_weights = None

    def _to_image(self, x):
        x = x.float()
        h = w = self.patch_size
        c = self.in_channels
        if x.dim() == 2:
            x = x.view(x.size(0), h, w, c).permute(0, 3, 1, 2).contiguous()
        return x

    def forward(self, x):
        return self.classifier(self.features(self._to_image(x)))

    def _loss(self, logits, y):
        return F.cross_entropy(logits, y, weight=self.class_weights)

    def training_step(self, batch, _):
        x, y = batch
        loss = self._loss(self(x), y.long())
        self.log("train_loss", loss, prog_bar=True)
        return loss

    def validation_step(self, batch, _):
        x, y = batch
        logits = self(x)
        loss = self._loss(logits, y.long())
        acc = (logits.argmax(1) == y).float().mean()
        self.log("val_loss", loss, prog_bar=True)
        self.log("val_acc", acc, prog_bar=True)

    def test_step(self, batch, _):
        x, y = batch
        logits = self(x)
        loss = self._loss(logits, y.long())
        acc = (logits.argmax(1) == y).float().mean()
        self.log("test_loss", loss)
        self.log("test_acc", acc, prog_bar=True)

    def configure_optimizers(self):
        opt = torch.optim.AdamW(self.parameters(), lr=self.lr, weight_decay=1e-4)
        sched = torch.optim.lr_scheduler.CosineAnnealingLR(opt, T_max=self.max_epochs)
        return {"optimizer": opt, "lr_scheduler": sched}

# ============================================================
# Load data -- ALL 16 bands (4 acquisitions x 4 bands)
# ============================================================
FINAL_DIR = Path("/p/scratch/training2600/TeamGudnason/training_data/final_geo_col")

X_train_raw = np.load(FINAL_DIR / "patches_train.npz")['patches']  # (N, 3, 3, 16)
y_train_raw = np.load(FINAL_DIR / "labels_train.npy")
X_val_raw   = np.load(FINAL_DIR / "patches_val.npz")['patches']
y_val_raw   = np.load(FINAL_DIR / "labels_val.npy")
X_test_raw  = np.load(FINAL_DIR / "patches_test.npz")['patches']
y_test_raw  = np.load(FINAL_DIR / "labels_test.npy")

print(f"Patch shape: {X_train_raw.shape} -- using all 16 bands (time series)")

# Normalize
X_all_tr = (X_train_raw.astype(np.float32) * 0.0001).reshape(X_train_raw.shape[0], -1)
X_all_va = (X_val_raw.astype(np.float32)   * 0.0001).reshape(X_val_raw.shape[0], -1)
X_all_te = (X_test_raw.astype(np.float32)  * 0.0001).reshape(X_test_raw.shape[0], -1)

# Remap labels
unique_all = np.unique(y_train_raw)
label_to_idx  = {int(lbl): i for i, lbl in enumerate(unique_all)}
idx_to_corine = {i: int(lbl) for i, lbl in enumerate(unique_all)}

y_all_tr = np.array([label_to_idx[int(l)] for l in y_train_raw], dtype=np.int64)
y_all_va = np.array([label_to_idx.get(int(l), -1) for l in y_val_raw], dtype=np.int64)
y_all_te = np.array([label_to_idx.get(int(l), -1) for l in y_test_raw], dtype=np.int64)

X_all_va, y_all_va = X_all_va[y_all_va != -1], y_all_va[y_all_va != -1]
X_all_te, y_all_te = X_all_te[y_all_te != -1], y_all_te[y_all_te != -1]

# Drop < 200 samples
classes_full, counts_full = np.unique(y_all_tr, return_counts=True)
valid = set(classes_full[counts_full >= 200])
mask_tr = np.isin(y_all_tr, list(valid))
mask_va = np.isin(y_all_va, list(valid))
mask_te = np.isin(y_all_te, list(valid))
X_all_tr, y_all_tr = X_all_tr[mask_tr], y_all_tr[mask_tr]
X_all_va, y_all_va = X_all_va[mask_va], y_all_va[mask_va]
X_all_te, y_all_te = X_all_te[mask_te], y_all_te[mask_te]

# Contiguous remap
unique_final = np.unique(y_all_tr)
final_map = {int(l): i for i, l in enumerate(unique_final)}
y_all_tr = np.array([final_map[l] for l in y_all_tr], dtype=np.int64)
y_all_va = np.array([final_map[l] for l in y_all_va], dtype=np.int64)
y_all_te = np.array([final_map[l] for l in y_all_te], dtype=np.int64)

# Top 4 classes
classes_17, counts_17 = np.unique(y_all_tr, return_counts=True)
top4_idx = classes_17[np.argsort(counts_17)[::-1][:4]]
print(f"Top 4: {[idx_to_corine[unique_final[int(i)]] for i in top4_idx]}")

mask_tr = np.isin(y_all_tr, top4_idx)
mask_va = np.isin(y_all_va, top4_idx)
mask_te = np.isin(y_all_te, top4_idx)

X_train, y_tr = X_all_tr[mask_tr], y_all_tr[mask_tr]
X_val,   y_va = X_all_va[mask_va], y_all_va[mask_va]
X_test,  y_te = X_all_te[mask_te], y_all_te[mask_te]

top4_map = {int(c): i for i, c in enumerate(sorted(top4_idx))}
y_train = np.array([top4_map[l] for l in y_tr], dtype=np.int64)
y_val   = np.array([top4_map[l] for l in y_va], dtype=np.int64)
y_test  = np.array([top4_map[l] for l in y_te], dtype=np.int64)

num_classes = 4
CLASS_NAMES = {i: f"CORINE {idx_to_corine[unique_final[int(sorted(top4_idx)[i])]]}"
               for i in range(4)}

classes_k, counts_k = np.unique(y_train, return_counts=True)
print(f"\nTrain: {X_train.shape[0]}, Val: {X_val.shape[0]}, Test: {X_test.shape[0]}")
for c, n in zip(classes_k, counts_k):
    print(f"  {CLASS_NAMES[c]}: {n} ({100*n/len(y_train):.1f}%)")

# Class weights
raw_weights = np.array([1.0/counts_k[c] for c in range(num_classes)], dtype=np.float64)
sqrt_weights = np.sqrt(raw_weights)
sqrt_weights /= sqrt_weights.sum() / num_classes
class_weights_t = torch.tensor(sqrt_weights, dtype=torch.float32)

# Dataloaders
train_dataset = YourCustomDataset(X_train, y_train)
val_dataset   = YourCustomDataset(X_val,   y_val)
test_dataset  = YourCustomDataset(X_test,  y_test)

train_dataloader = DataLoader(train_dataset, batch_size=512, shuffle=True,
                               num_workers=4, pin_memory=False)
val_dataloader   = DataLoader(val_dataset,   batch_size=512, shuffle=False,
                               num_workers=4, pin_memory=False)
test_dataloader  = DataLoader(test_dataset,  batch_size=512, shuffle=False,
                               num_workers=4, pin_memory=False)

# Model -- 16 in_channels fyrir time series
patch_size  = 3
in_channels = 16
max_epochs  = 200

model_ts = ConvNet(
    lr=3e-4, num_classes=4, in_channels=in_channels,
    patch_size=patch_size, class_weights=class_weights_t, max_epochs=max_epochs,
)

early_stop = pl.callbacks.EarlyStopping(monitor="val_loss", patience=20, mode="min")

trainer_ts = pl.Trainer(
    accelerator="gpu", devices=1, max_epochs=max_epochs,
    gradient_clip_val=1.0, log_every_n_steps=10,
    enable_progress_bar=True, callbacks=[early_stop],
)

trainer_ts.fit(model_ts, train_dataloader, val_dataloader)
trainer_ts.test(model_ts, dataloaders=test_dataloader)

# Diagnostics
model_ts.eval()
model_ts = model_ts.cuda()
preds_all, targets_all = [], []
with torch.no_grad():
    for xb, yb in test_dataloader:
        logits = model_ts(xb.cuda())
        preds_all.append(logits.argmax(1).cpu().numpy())
        targets_all.append(yb.numpy())

y_true = np.concatenate(targets_all).astype(np.int64)
y_pred = np.concatenate(preds_all).astype(np.int64)

cm = np.zeros((4, 4), dtype=np.int64)
for t, p in zip(y_true, y_pred):
    cm[t, p] += 1

tp        = np.diag(cm).astype(np.float64)
support   = cm.sum(axis=1).astype(np.float64)
pred_c    = cm.sum(axis=0).astype(np.float64)
recall    = np.divide(tp, support, out=np.zeros_like(tp), where=support > 0)
precision = np.divide(tp, pred_c,  out=np.zeros_like(tp), where=pred_c > 0)
f1        = np.divide(2*precision*recall, precision+recall,
                      out=np.zeros_like(tp), where=(precision+recall) > 0)

print(f"\n========== RESULTS (16 bands, geo split) ==========")
print(f"Overall accuracy:  {tp.sum()/cm.sum():.4f}")
print(f"Balanced accuracy: {recall.mean():.4f}")
print(f"Macro-F1:          {f1.mean():.4f}")
print(f"\nPer-class:")
for i in range(4):
    print(f"  {CLASS_NAMES[i]:<20} recall={recall[i]:.3f} f1={f1[i]:.3f} support={int(support[i])}")
print(f"\nConfusion matrix:\n{cm}")
print(f"\nPredicted distribution: {np.unique(y_pred, return_counts=True)}")

Patch shape: (100000, 3, 3, 16) -- using all 16 bands (time series)
Top 4: [12, 23, 41, 18]

Train: 76506, Val: 23385, Test: 19826
  CORINE 12: 44778 (58.5%)
  CORINE 18: 5456 (7.1%)
  CORINE 23: 20197 (26.4%)
  CORINE 41: 6075 (7.9%)


GPU available: True (cuda), used: True
TPU available: False, using: 0 TPU cores
💡 Tip: For seamless cloud logging and experiment tracking, try installing [litlogger](https://pypi.org/project/litlogger/) to enable LitLogger, which logs metrics and artifacts automatically to the Lightning Experiments platform.
💡 Tip: For seamless cloud uploads and versioning, try installing [litmodels](https://pypi.org/project/litmodels/) to enable LitModelCheckpoint, which syncs automatically with the Lightning model registry.
/p/project1/training2600/TeamGudnason/envs/danube/lib/python3.12/site-packages/lightning/pytorch/callbacks/model_checkpoint.py:881: Checkpoint directory /p/project1/training2600/gudnason2/repos/EO_danube_river/lightning_logs/version_14664797/checkpoints exists and is not empty.
LOCAL_RANK: 0 - CUDA_VISIBLE_DEVICES: [0]


┏━━━┳━━━━━━━━━━━━┳━━━━━━━━━━━━┳━━━━━━━━┳━━━━━━━┳━━━━━━━┓
┃   ┃ Name       ┃ Type       ┃ Params ┃ Mode  ┃ FLOPs ┃
┡━━━╇━━━━━━━━━━━━╇━━━━━━━━━━━━╇━━━━━━━━╇━━━━━━━╇━━━━━━━┩
│ 0 │ features   │ Sequential │ 83.5 K │ train │     0 │
│ 1 │ classifier │ Sequential │  8.5 K │ train │     0 │
└───┴────────────┴────────────┴────────┴───────┴───────┘

Trainable params: 92.0 K                                                                                           
Non-trainable params: 0                                                                                            
Total params: 92.0 K                                                                                               
Total estimated model params size (MB): 0                                                                          
Modules in train mode: 14                                                                                          
Modules in eval mode: 0                                                                                            
Total FLOPs: 0

SLURM auto-requeueing enabled. Setting signal handlers.


Output()

LOCAL_RANK: 0 - CUDA_VISIBLE_DEVICES: [0]
SLURM auto-requeueing enabled. Setting signal handlers.


Output()

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━━━━┓
┃        Test metric        ┃       DataLoader 0        ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━━━━┩
│         test_acc          │    0.6761828064918518     │
│         test_loss         │    25.393238067626953     │
└───────────────────────────┴───────────────────────────┘


========== RESULTS (16 bands, geo split) ==========
Overall accuracy:  0.6762
Balanced accuracy: 0.2500
Macro-F1:          0.2017

Per-class:
  CORINE 12            recall=1.000 f1=0.807 support=13406
  CORINE 18            recall=0.000 f1=0.000 support=1648
  CORINE 23            recall=0.000 f1=0.000 support=4470
  CORINE 41            recall=0.000 f1=0.000 support=302

Confusion matrix:
[[13406     0     0     0]
 [ 1648     0     0     0]
 [ 4470     0     0     0]
 [  302     0     0     0]]

Predicted distribution: (array([0]), array([19826]))


## Class Distribution Analysis: Column Split Mismatch

Compares class distribution between train (left columns) and test (right columns).
Urban fabric (CORINE 2) is 4.8% in train but 12.3% in test.
Water bodies (CORINE 41) is 6.1% in train but only 1.0% in test.
This mismatch confirms that vertical column splits produce spatially biased
evaluation and motivated the switch to block cross-validation.

In [5]:
print("Train label distribution:")
tr_cls, tr_cnt = np.unique(y_train_raw, return_counts=True)
for c, n in zip(tr_cls, tr_cnt):
    print(f"  CORINE {c}: {n} ({100*n/len(y_train_raw):.1f}%)")

print("\nTest label distribution:")
te_cls, te_cnt = np.unique(y_test_raw, return_counts=True)
for c, n in zip(te_cls, te_cnt):
    print(f"  CORINE {c}: {n} ({100*n/len(y_test_raw):.1f}%)")

Train label distribution:
  CORINE 2: 4798 (4.8%)
  CORINE 3: 1100 (1.1%)
  CORINE 4: 79 (0.1%)
  CORINE 5: 4 (0.0%)
  CORINE 6: 96 (0.1%)
  CORINE 7: 136 (0.1%)
  CORINE 8: 91 (0.1%)
  CORINE 9: 32 (0.0%)
  CORINE 10: 72 (0.1%)
  CORINE 11: 992 (1.0%)
  CORINE 12: 44778 (44.8%)
  CORINE 15: 1403 (1.4%)
  CORINE 16: 462 (0.5%)
  CORINE 18: 5456 (5.5%)
  CORINE 20: 2796 (2.8%)
  CORINE 21: 2021 (2.0%)
  CORINE 23: 20197 (20.2%)
  CORINE 24: 980 (1.0%)
  CORINE 25: 1351 (1.4%)
  CORINE 26: 2484 (2.5%)
  CORINE 29: 3374 (3.4%)
  CORINE 35: 616 (0.6%)
  CORINE 36: 92 (0.1%)
  CORINE 40: 515 (0.5%)
  CORINE 41: 6075 (6.1%)

Test label distribution:
  CORINE 1: 147 (0.5%)
  CORINE 2: 3679 (12.3%)
  CORINE 3: 1063 (3.5%)
  CORINE 4: 235 (0.8%)
  CORINE 5: 33 (0.1%)
  CORINE 6: 89 (0.3%)
  CORINE 7: 59 (0.2%)
  CORINE 8: 23 (0.1%)
  CORINE 9: 31 (0.1%)
  CORINE 10: 127 (0.4%)
  CORINE 11: 766 (2.6%)
  CORINE 12: 13406 (44.7%)
  CORINE 15: 106 (0.4%)
  CORINE 16: 296 (1.0%)
  CORINE 18: 1648 (5

In [ ]:
## CNN with Block Cross-Validation, 3×3 Patches, Weighted Sampler

First experiment using block cross-validation data (final_block directory).
Uses all 16 bands with weighted sampler. Achieves 21.5% overall accuracy and
24.7% balanced accuracy -- better than column split but still poor, as the
model predicts mostly CORINE 23 (forest). This motivated switching to larger
64×64 patches with the ResNet architecture.

In [6]:
import os
os.environ["CUDA_LAUNCH_BLOCKING"] = "1"
os.environ["CUDA_VISIBLE_DEVICES"] = "0"

import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
import lightning.pytorch as pl
from torch.utils.data import DataLoader, Dataset, WeightedRandomSampler
from pathlib import Path

class YourCustomDataset(Dataset):
    def __init__(self, data, labels):
        self.data = data
        self.labels = labels
    def __len__(self):
        return len(self.data)
    def __getitem__(self, idx):
        return self.data[idx], self.labels[idx]

class ConvNet(pl.LightningModule):
    def __init__(self, num_classes=4, in_channels=16, patch_size=3,
                 lr=3e-4, max_epochs=200, class_weights=None):
        super().__init__()
        self.lr = lr
        self.in_channels = in_channels
        self.patch_size = patch_size
        self.num_classes = num_classes
        self.max_epochs = max_epochs
        self.features = nn.Sequential(
            nn.Conv2d(in_channels, 64, kernel_size=3, padding=1),
            nn.BatchNorm2d(64), nn.ReLU(inplace=True),
            nn.Conv2d(64, 128, kernel_size=3, padding=1),
            nn.BatchNorm2d(128), nn.ReLU(inplace=True),
            nn.AdaptiveAvgPool2d((1, 1)),
        )
        self.classifier = nn.Sequential(
            nn.Flatten(), nn.Linear(128, 64),
            nn.ReLU(), nn.Dropout(0.3),
            nn.Linear(64, num_classes),
        )
        if class_weights is not None:
            self.register_buffer("class_weights", class_weights.float())
        else:
            self.class_weights = None

    def _to_image(self, x):
        x = x.float()
        h = w = self.patch_size
        c = self.in_channels
        if x.dim() == 2:
            x = x.view(x.size(0), h, w, c).permute(0, 3, 1, 2).contiguous()
        return x

    def forward(self, x):
        return self.classifier(self.features(self._to_image(x)))

    def _loss(self, logits, y):
        return F.cross_entropy(logits, y, weight=self.class_weights)

    def training_step(self, batch, _):
        x, y = batch
        loss = self._loss(self(x), y.long())
        self.log("train_loss", loss, prog_bar=True)
        return loss

    def validation_step(self, batch, _):
        x, y = batch
        logits = self(x)
        loss = self._loss(logits, y.long())
        acc = (logits.argmax(1) == y).float().mean()
        self.log("val_loss", loss, prog_bar=True)
        self.log("val_acc", acc, prog_bar=True)

    def test_step(self, batch, _):
        x, y = batch
        logits = self(x)
        loss = self._loss(logits, y.long())
        acc = (logits.argmax(1) == y).float().mean()
        self.log("test_loss", loss)
        self.log("test_acc", acc, prog_bar=True)

    def configure_optimizers(self):
        opt = torch.optim.AdamW(self.parameters(), lr=self.lr, weight_decay=1e-4)
        sched = torch.optim.lr_scheduler.CosineAnnealingLR(opt, T_max=self.max_epochs)
        return {"optimizer": opt, "lr_scheduler": sched}

# ============================================================
# Load data -- ALL 16 bands
# ============================================================
FINAL_DIR = Path("/p/scratch/training2600/TeamGudnason/training_data/final_block")

X_train_raw = np.load(FINAL_DIR / "patches_train.npz")['patches']
y_train_raw = np.load(FINAL_DIR / "labels_train.npy")
X_val_raw   = np.load(FINAL_DIR / "patches_val.npz")['patches']
y_val_raw   = np.load(FINAL_DIR / "labels_val.npy")
X_test_raw  = np.load(FINAL_DIR / "patches_test.npz")['patches']
y_test_raw  = np.load(FINAL_DIR / "labels_test.npy")

print(f"Patch shape: {X_train_raw.shape}")

X_all_tr = (X_train_raw.astype(np.float32) * 0.0001).reshape(X_train_raw.shape[0], -1)
X_all_va = (X_val_raw.astype(np.float32)   * 0.0001).reshape(X_val_raw.shape[0], -1)
X_all_te = (X_test_raw.astype(np.float32)  * 0.0001).reshape(X_test_raw.shape[0], -1)

unique_all    = np.unique(y_train_raw)
label_to_idx  = {int(lbl): i for i, lbl in enumerate(unique_all)}
idx_to_corine = {i: int(lbl) for i, lbl in enumerate(unique_all)}

y_all_tr = np.array([label_to_idx[int(l)] for l in y_train_raw], dtype=np.int64)
y_all_va = np.array([label_to_idx.get(int(l), -1) for l in y_val_raw], dtype=np.int64)
y_all_te = np.array([label_to_idx.get(int(l), -1) for l in y_test_raw], dtype=np.int64)

X_all_va, y_all_va = X_all_va[y_all_va != -1], y_all_va[y_all_va != -1]
X_all_te, y_all_te = X_all_te[y_all_te != -1], y_all_te[y_all_te != -1]

classes_full, counts_full = np.unique(y_all_tr, return_counts=True)
valid = set(classes_full[counts_full >= 200])
mask_tr = np.isin(y_all_tr, list(valid))
mask_va = np.isin(y_all_va, list(valid))
mask_te = np.isin(y_all_te, list(valid))
X_all_tr, y_all_tr = X_all_tr[mask_tr], y_all_tr[mask_tr]
X_all_va, y_all_va = X_all_va[mask_va], y_all_va[mask_va]
X_all_te, y_all_te = X_all_te[mask_te], y_all_te[mask_te]

unique_final = np.unique(y_all_tr)
final_map = {int(l): i for i, l in enumerate(unique_final)}
y_all_tr = np.array([final_map[l] for l in y_all_tr], dtype=np.int64)
y_all_va = np.array([final_map[l] for l in y_all_va], dtype=np.int64)
y_all_te = np.array([final_map[l] for l in y_all_te], dtype=np.int64)

classes_17, counts_17 = np.unique(y_all_tr, return_counts=True)
top4_idx = classes_17[np.argsort(counts_17)[::-1][:4]]
print(f"Top 4 CORINE: {[idx_to_corine[unique_final[int(i)]] for i in top4_idx]}")

mask_tr = np.isin(y_all_tr, top4_idx)
mask_va = np.isin(y_all_va, top4_idx)
mask_te = np.isin(y_all_te, top4_idx)

X_train, y_tr = X_all_tr[mask_tr], y_all_tr[mask_tr]
X_val,   y_va = X_all_va[mask_va], y_all_va[mask_va]
X_test,  y_te = X_all_te[mask_te], y_all_te[mask_te]

top4_map = {int(c): i for i, c in enumerate(sorted(top4_idx))}
y_train = np.array([top4_map[l] for l in y_tr], dtype=np.int64)
y_val   = np.array([top4_map[l] for l in y_va], dtype=np.int64)
y_test  = np.array([top4_map[l] for l in y_te], dtype=np.int64)

num_classes = 4
CLASS_NAMES = {i: f"CORINE {idx_to_corine[unique_final[int(sorted(top4_idx)[i])]]}"
               for i in range(4)}

classes_k, counts_k = np.unique(y_train, return_counts=True)
print(f"\nTrain: {X_train.shape[0]}, Val: {X_val.shape[0]}, Test: {X_test.shape[0]}")
for c, n in zip(classes_k, counts_k):
    print(f"  {CLASS_NAMES[c]}: {n} ({100*n/len(y_train):.1f}%)")

# ============================================================
# Weighted sampler -- í staðinn fyrir class weights
# ============================================================
sample_weights = np.array([1.0/counts_k[l] for l in y_train], dtype=np.float64)
sampler = WeightedRandomSampler(
    weights=torch.as_tensor(sample_weights, dtype=torch.double),
    num_samples=len(sample_weights),
    replacement=True,
)
print(f"\nUsing weighted sampler -- each class equally represented in batches")

# ============================================================
# Dataloaders
# ============================================================
train_dataset = YourCustomDataset(X_train, y_train)
val_dataset   = YourCustomDataset(X_val,   y_val)
test_dataset  = YourCustomDataset(X_test,  y_test)

train_dataloader = DataLoader(train_dataset, batch_size=512,
                               sampler=sampler, shuffle=False,
                               num_workers=4, pin_memory=False)
val_dataloader   = DataLoader(val_dataset,   batch_size=512, shuffle=False,
                               num_workers=4, pin_memory=False)
test_dataloader  = DataLoader(test_dataset,  batch_size=512, shuffle=False,
                               num_workers=4, pin_memory=False)

# ============================================================
# Model -- engar class weights, weighted sampler tekur við
# ============================================================
patch_size  = 3
in_channels = 16
max_epochs  = 200

model_ts = ConvNet(
    lr=3e-4, num_classes=4, in_channels=in_channels,
    patch_size=patch_size, class_weights=None,
    max_epochs=max_epochs,
)

early_stop = pl.callbacks.EarlyStopping(monitor="val_loss", patience=20, mode="min")

trainer_ts = pl.Trainer(
    accelerator="gpu", devices=1, max_epochs=max_epochs,
    gradient_clip_val=1.0, log_every_n_steps=10,
    enable_progress_bar=True, callbacks=[early_stop],
)

trainer_ts.fit(model_ts, train_dataloader, val_dataloader)
trainer_ts.test(model_ts, dataloaders=test_dataloader)

# ============================================================
# Diagnostics
# ============================================================
model_ts.eval()
model_ts = model_ts.cuda()
preds_all, targets_all = [], []
with torch.no_grad():
    for xb, yb in test_dataloader:
        logits = model_ts(xb.cuda())
        preds_all.append(logits.argmax(1).cpu().numpy())
        targets_all.append(yb.numpy())

y_true = np.concatenate(targets_all).astype(np.int64)
y_pred = np.concatenate(preds_all).astype(np.int64)

cm = np.zeros((4, 4), dtype=np.int64)
for t, p in zip(y_true, y_pred):
    cm[t, p] += 1

tp        = np.diag(cm).astype(np.float64)
support   = cm.sum(axis=1).astype(np.float64)
pred_c    = cm.sum(axis=0).astype(np.float64)
recall    = np.divide(tp, support, out=np.zeros_like(tp), where=support > 0)
precision = np.divide(tp, pred_c,  out=np.zeros_like(tp), where=pred_c > 0)
f1        = np.divide(2*precision*recall, precision+recall,
                      out=np.zeros_like(tp), where=(precision+recall) > 0)

print(f"\n========== RESULTS (16 bands, weighted sampler, geo split) ==========")
print(f"Overall accuracy:  {tp.sum()/cm.sum():.4f}")
print(f"Balanced accuracy: {recall.mean():.4f}")
print(f"Macro-F1:          {f1.mean():.4f}")
print(f"\nPer-class:")
for i in range(4):
    print(f"  {CLASS_NAMES[i]:<20} recall={recall[i]:.3f} f1={f1[i]:.3f} support={int(support[i])}")
print(f"\nConfusion matrix:\n{cm}")
print(f"\nPredicted distribution: {np.unique(y_pred, return_counts=True)}")

Patch shape: (100000, 3, 3, 16)
Top 4 CORINE: [12, 23, 2, 18]

Train: 77907, Val: 23158, Test: 23113
  CORINE 2: 6391 (8.2%)
  CORINE 12: 47138 (60.5%)
  CORINE 18: 5647 (7.2%)
  CORINE 23: 18731 (24.0%)

Using weighted sampler -- each class equally represented in batches


GPU available: True (cuda), used: True
TPU available: False, using: 0 TPU cores
💡 Tip: For seamless cloud logging and experiment tracking, try installing [litlogger](https://pypi.org/project/litlogger/) to enable LitLogger, which logs metrics and artifacts automatically to the Lightning Experiments platform.
💡 Tip: For seamless cloud uploads and versioning, try installing [litmodels](https://pypi.org/project/litmodels/) to enable LitModelCheckpoint, which syncs automatically with the Lightning model registry.
LOCAL_RANK: 0 - CUDA_VISIBLE_DEVICES: [0]


┏━━━┳━━━━━━━━━━━━┳━━━━━━━━━━━━┳━━━━━━━━┳━━━━━━━┳━━━━━━━┓
┃   ┃ Name       ┃ Type       ┃ Params ┃ Mode  ┃ FLOPs ┃
┡━━━╇━━━━━━━━━━━━╇━━━━━━━━━━━━╇━━━━━━━━╇━━━━━━━╇━━━━━━━┩
│ 0 │ features   │ Sequential │ 83.5 K │ train │     0 │
│ 1 │ classifier │ Sequential │  8.5 K │ train │     0 │
└───┴────────────┴────────────┴────────┴───────┴───────┘

Trainable params: 92.0 K                                                                                           
Non-trainable params: 0                                                                                            
Total params: 92.0 K                                                                                               
Total estimated model params size (MB): 0                                                                          
Modules in train mode: 14                                                                                          
Modules in eval mode: 0                                                                                            
Total FLOPs: 0

SLURM auto-requeueing enabled. Setting signal handlers.


Output()

LOCAL_RANK: 0 - CUDA_VISIBLE_DEVICES: [0]
SLURM auto-requeueing enabled. Setting signal handlers.


Output()

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━━━━┓
┃        Test metric        ┃       DataLoader 0        ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━━━━┩
│         test_acc          │    0.21507376432418823    │
│         test_loss         │    52.057559967041016     │
└───────────────────────────┴───────────────────────────┘


========== RESULTS (16 bands, weighted sampler, geo split) ==========
Overall accuracy:  0.2151
Balanced accuracy: 0.2467
Macro-F1:          0.0900

Per-class:
  CORINE 2             recall=0.000 f1=0.000 support=2038
  CORINE 12            recall=0.000 f1=0.000 support=14303
  CORINE 18            recall=0.003 f1=0.005 support=1724
  CORINE 23            recall=0.984 f1=0.355 support=5048

Confusion matrix:
[[    0     0    15  2023]
 [    0     0    70 14233]
 [    0     0     5  1719]
 [    0     0    82  4966]]

Predicted distribution: (array([2, 3]), array([  172, 22941]))


## CNN with Block Split, 3×3 Patches, Class Weights

Trains CNN using block cross-validation data with sqrt-scaled class weights.
Achieves only 8.8% overall accuracy -- the model predicts only CORINE 2 (urban),
the opposite problem from weighted sampler. Neither class weights nor weighted
sampler can compensate for insufficient spatial context in 3×3 patches.
This confirmed that larger patches were needed.

In [7]:
import os
os.environ["CUDA_LAUNCH_BLOCKING"] = "1"
os.environ["CUDA_VISIBLE_DEVICES"] = "0"

import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
import lightning.pytorch as pl
from torch.utils.data import DataLoader, Dataset
from pathlib import Path

class YourCustomDataset(Dataset):
    def __init__(self, data, labels):
        self.data = data
        self.labels = labels
    def __len__(self):
        return len(self.data)
    def __getitem__(self, idx):
        return self.data[idx], self.labels[idx]

class ConvNet(pl.LightningModule):
    def __init__(self, num_classes=4, in_channels=16, patch_size=3,
                 lr=3e-4, max_epochs=200, class_weights=None):
        super().__init__()
        self.lr = lr
        self.in_channels = in_channels
        self.patch_size = patch_size
        self.num_classes = num_classes
        self.max_epochs = max_epochs
        self.features = nn.Sequential(
            nn.Conv2d(in_channels, 64, kernel_size=3, padding=1),
            nn.BatchNorm2d(64), nn.ReLU(inplace=True),
            nn.Conv2d(64, 128, kernel_size=3, padding=1),
            nn.BatchNorm2d(128), nn.ReLU(inplace=True),
            nn.AdaptiveAvgPool2d((1, 1)),
        )
        self.classifier = nn.Sequential(
            nn.Flatten(), nn.Linear(128, 64),
            nn.ReLU(), nn.Dropout(0.3),
            nn.Linear(64, num_classes),
        )
        if class_weights is not None:
            self.register_buffer("class_weights", class_weights.float())
        else:
            self.class_weights = None

    def _to_image(self, x):
        x = x.float()
        h = w = self.patch_size
        c = self.in_channels
        if x.dim() == 2:
            x = x.view(x.size(0), h, w, c).permute(0, 3, 1, 2).contiguous()
        return x

    def forward(self, x):
        return self.classifier(self.features(self._to_image(x)))

    def _loss(self, logits, y):
        return F.cross_entropy(logits, y, weight=self.class_weights)

    def training_step(self, batch, _):
        x, y = batch
        loss = self._loss(self(x), y.long())
        self.log("train_loss", loss, prog_bar=True)
        return loss

    def validation_step(self, batch, _):
        x, y = batch
        logits = self(x)
        loss = self._loss(logits, y.long())
        acc = (logits.argmax(1) == y).float().mean()
        self.log("val_loss", loss, prog_bar=True)
        self.log("val_acc", acc, prog_bar=True)

    def test_step(self, batch, _):
        x, y = batch
        logits = self(x)
        loss = self._loss(logits, y.long())
        acc = (logits.argmax(1) == y).float().mean()
        self.log("test_loss", loss)
        self.log("test_acc", acc, prog_bar=True)

    def configure_optimizers(self):
        opt = torch.optim.AdamW(self.parameters(), lr=self.lr, weight_decay=1e-4)
        sched = torch.optim.lr_scheduler.CosineAnnealingLR(opt, T_max=self.max_epochs)
        return {"optimizer": opt, "lr_scheduler": sched}

# ============================================================
# Load data
# ============================================================
FINAL_DIR = Path("/p/scratch/training2600/TeamGudnason/training_data/final_block")

X_train_raw = np.load(FINAL_DIR / "patches_train.npz")['patches']
y_train_raw = np.load(FINAL_DIR / "labels_train.npy")
X_val_raw   = np.load(FINAL_DIR / "patches_val.npz")['patches']
y_val_raw   = np.load(FINAL_DIR / "labels_val.npy")
X_test_raw  = np.load(FINAL_DIR / "patches_test.npz")['patches']
y_test_raw  = np.load(FINAL_DIR / "labels_test.npy")

print(f"Patch shape: {X_train_raw.shape}")

X_all_tr = (X_train_raw.astype(np.float32) * 0.0001).reshape(X_train_raw.shape[0], -1)
X_all_va = (X_val_raw.astype(np.float32)   * 0.0001).reshape(X_val_raw.shape[0], -1)
X_all_te = (X_test_raw.astype(np.float32)  * 0.0001).reshape(X_test_raw.shape[0], -1)

unique_all    = np.unique(y_train_raw)
label_to_idx  = {int(lbl): i for i, lbl in enumerate(unique_all)}
idx_to_corine = {i: int(lbl) for i, lbl in enumerate(unique_all)}

y_all_tr = np.array([label_to_idx[int(l)] for l in y_train_raw], dtype=np.int64)
y_all_va = np.array([label_to_idx.get(int(l), -1) for l in y_val_raw], dtype=np.int64)
y_all_te = np.array([label_to_idx.get(int(l), -1) for l in y_test_raw], dtype=np.int64)

X_all_va, y_all_va = X_all_va[y_all_va != -1], y_all_va[y_all_va != -1]
X_all_te, y_all_te = X_all_te[y_all_te != -1], y_all_te[y_all_te != -1]

classes_full, counts_full = np.unique(y_all_tr, return_counts=True)
valid = set(classes_full[counts_full >= 200])
mask_tr = np.isin(y_all_tr, list(valid))
mask_va = np.isin(y_all_va, list(valid))
mask_te = np.isin(y_all_te, list(valid))
X_all_tr, y_all_tr = X_all_tr[mask_tr], y_all_tr[mask_tr]
X_all_va, y_all_va = X_all_va[mask_va], y_all_va[mask_va]
X_all_te, y_all_te = X_all_te[mask_te], y_all_te[mask_te]

unique_final = np.unique(y_all_tr)
final_map = {int(l): i for i, l in enumerate(unique_final)}
y_all_tr = np.array([final_map[l] for l in y_all_tr], dtype=np.int64)
y_all_va = np.array([final_map[l] for l in y_all_va], dtype=np.int64)
y_all_te = np.array([final_map[l] for l in y_all_te], dtype=np.int64)

# Top 4 classes
classes_17, counts_17 = np.unique(y_all_tr, return_counts=True)
top4_idx = classes_17[np.argsort(counts_17)[::-1][:4]]
print(f"Top 4 CORINE: {[idx_to_corine[unique_final[int(i)]] for i in top4_idx]}")

mask_tr = np.isin(y_all_tr, top4_idx)
mask_va = np.isin(y_all_va, top4_idx)
mask_te = np.isin(y_all_te, top4_idx)

X_train, y_tr = X_all_tr[mask_tr], y_all_tr[mask_tr]
X_val,   y_va = X_all_va[mask_va], y_all_va[mask_va]
X_test,  y_te = X_all_te[mask_te], y_all_te[mask_te]

top4_map = {int(c): i for i, c in enumerate(sorted(top4_idx))}
y_train = np.array([top4_map[l] for l in y_tr], dtype=np.int64)
y_val   = np.array([top4_map[l] for l in y_va], dtype=np.int64)
y_test  = np.array([top4_map[l] for l in y_te], dtype=np.int64)

num_classes = 4
CLASS_NAMES = {i: f"CORINE {idx_to_corine[unique_final[int(sorted(top4_idx)[i])]]}"
               for i in range(4)}

classes_k, counts_k = np.unique(y_train, return_counts=True)
print(f"\nTrain: {X_train.shape[0]}, Val: {X_val.shape[0]}, Test: {X_test.shape[0]}")
for c, n in zip(classes_k, counts_k):
    print(f"  {CLASS_NAMES[c]}: {n} ({100*n/len(y_train):.1f}%)")

# ============================================================
# Class weights (sqrt scaled)
# ============================================================
raw_weights = np.array([1.0/counts_k[c] for c in range(num_classes)], dtype=np.float64)
sqrt_weights = np.sqrt(raw_weights)
sqrt_weights /= sqrt_weights.sum() / num_classes
class_weights_t = torch.tensor(sqrt_weights, dtype=torch.float32)
print(f"\nClass weights: {class_weights_t}")

# ============================================================
# Dataloaders
# ============================================================
train_dataset = YourCustomDataset(X_train, y_train)
val_dataset   = YourCustomDataset(X_val,   y_val)
test_dataset  = YourCustomDataset(X_test,  y_test)

train_dataloader = DataLoader(train_dataset, batch_size=512, shuffle=True,
                               num_workers=4, pin_memory=False)
val_dataloader   = DataLoader(val_dataset,   batch_size=512, shuffle=False,
                               num_workers=4, pin_memory=False)
test_dataloader  = DataLoader(test_dataset,  batch_size=512, shuffle=False,
                               num_workers=4, pin_memory=False)

# ============================================================
# Model + Trainer
# ============================================================
patch_size  = 3
in_channels = 16
max_epochs  = 200

model_block = ConvNet(
    lr=3e-4, num_classes=4, in_channels=in_channels,
    patch_size=patch_size, class_weights=class_weights_t,
    max_epochs=max_epochs,
)

early_stop = pl.callbacks.EarlyStopping(monitor="val_loss", patience=20, mode="min")

trainer_block = pl.Trainer(
    accelerator="gpu", devices=1, max_epochs=max_epochs,
    gradient_clip_val=1.0, log_every_n_steps=10,
    enable_progress_bar=True, callbacks=[early_stop],
)

trainer_block.fit(model_block, train_dataloader, val_dataloader)
trainer_block.test(model_block, dataloaders=test_dataloader)

# ============================================================
# Diagnostics
# ============================================================
model_block.eval()
model_block = model_block.cuda()
preds_all, targets_all = [], []
with torch.no_grad():
    for xb, yb in test_dataloader:
        logits = model_block(xb.cuda())
        preds_all.append(logits.argmax(1).cpu().numpy())
        targets_all.append(yb.numpy())

y_true = np.concatenate(targets_all).astype(np.int64)
y_pred = np.concatenate(preds_all).astype(np.int64)

cm = np.zeros((4, 4), dtype=np.int64)
for t, p in zip(y_true, y_pred):
    cm[t, p] += 1

tp        = np.diag(cm).astype(np.float64)
support   = cm.sum(axis=1).astype(np.float64)
pred_c    = cm.sum(axis=0).astype(np.float64)
recall    = np.divide(tp, support, out=np.zeros_like(tp), where=support > 0)
precision = np.divide(tp, pred_c,  out=np.zeros_like(tp), where=pred_c > 0)
f1        = np.divide(2*precision*recall, precision+recall,
                      out=np.zeros_like(tp), where=(precision+recall) > 0)

print(f"\n========== RESULTS (block split, 16 bands, class weights) ==========")
print(f"Overall accuracy:  {tp.sum()/cm.sum():.4f}")
print(f"Balanced accuracy: {recall.mean():.4f}")
print(f"Macro-F1:          {f1.mean():.4f}")
print(f"\nPer-class:")
for i in range(4):
    print(f"  {CLASS_NAMES[i]:<20} recall={recall[i]:.3f} precision={precision[i]:.3f} f1={f1[i]:.3f} support={int(support[i])}")
print(f"\nConfusion matrix:\n{cm}")
print(f"\nPredicted distribution: {np.unique(y_pred, return_counts=True)}")

Patch shape: (100000, 3, 3, 16)
Top 4 CORINE: [12, 23, 2, 18]

Train: 77907, Val: 23158, Test: 23113
  CORINE 2: 6391 (8.2%)
  CORINE 12: 47138 (60.5%)
  CORINE 18: 5647 (7.2%)
  CORINE 23: 18731 (24.0%)

Class weights: tensor([1.3262, 0.4883, 1.4108, 0.7747])


GPU available: True (cuda), used: True
TPU available: False, using: 0 TPU cores
💡 Tip: For seamless cloud logging and experiment tracking, try installing [litlogger](https://pypi.org/project/litlogger/) to enable LitLogger, which logs metrics and artifacts automatically to the Lightning Experiments platform.
💡 Tip: For seamless cloud uploads and versioning, try installing [litmodels](https://pypi.org/project/litmodels/) to enable LitModelCheckpoint, which syncs automatically with the Lightning model registry.
LOCAL_RANK: 0 - CUDA_VISIBLE_DEVICES: [0]


┏━━━┳━━━━━━━━━━━━┳━━━━━━━━━━━━┳━━━━━━━━┳━━━━━━━┳━━━━━━━┓
┃   ┃ Name       ┃ Type       ┃ Params ┃ Mode  ┃ FLOPs ┃
┡━━━╇━━━━━━━━━━━━╇━━━━━━━━━━━━╇━━━━━━━━╇━━━━━━━╇━━━━━━━┩
│ 0 │ features   │ Sequential │ 83.5 K │ train │     0 │
│ 1 │ classifier │ Sequential │  8.5 K │ train │     0 │
└───┴────────────┴────────────┴────────┴───────┴───────┘

Trainable params: 92.0 K                                                                                           
Non-trainable params: 0                                                                                            
Total params: 92.0 K                                                                                               
Total estimated model params size (MB): 0                                                                          
Modules in train mode: 14                                                                                          
Modules in eval mode: 0                                                                                            
Total FLOPs: 0

SLURM auto-requeueing enabled. Setting signal handlers.


Output()

LOCAL_RANK: 0 - CUDA_VISIBLE_DEVICES: [0]
SLURM auto-requeueing enabled. Setting signal handlers.


Output()

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━━━━┓
┃        Test metric        ┃       DataLoader 0        ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━━━━┩
│         test_acc          │    0.08817548304796219    │
│         test_loss         │     80.84229278564453     │
└───────────────────────────┴───────────────────────────┘


========== RESULTS (block split, 16 bands, class weights) ==========
Overall accuracy:  0.0882
Balanced accuracy: 0.2500
Macro-F1:          0.0405

Per-class:
  CORINE 2             recall=1.000 precision=0.088 f1=0.162 support=2038
  CORINE 12            recall=0.000 precision=0.000 f1=0.000 support=14303
  CORINE 18            recall=0.000 precision=0.000 f1=0.000 support=1724
  CORINE 23            recall=0.000 precision=0.000 f1=0.000 support=5048

Confusion matrix:
[[ 2038     0     0     0]
 [14303     0     0     0]
 [ 1724     0     0     0]
 [ 5048     0     0     0]]

Predicted distribution: (array([0]), array([23113]))


## Final Model: ResNet 64×64 with Block Cross-Validation

Switches to a ResNet architecture operating on 64×64 pixel patches (640×640 m)
with 16 spectral bands (4 dates × 4 bands) and weighted random sampler.
Achieves 88.6% overall accuracy and 81.0% balanced accuracy under block
cross-validation -- a major improvement over all previous experiments.
Arable land (F1=0.927) and forest (F1=0.917) perform best; pastures remain
hardest (F1=0.545) due to spectral similarity with arable land.
This is the main result of the project.

In [2]:
import os
os.environ["CUDA_LAUNCH_BLOCKING"] = "1"
os.environ["CUDA_VISIBLE_DEVICES"] = "0"

import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
import lightning.pytorch as pl
from torch.utils.data import DataLoader, Dataset, WeightedRandomSampler
from pathlib import Path

# ============================================================
# Dataset
# ============================================================
class PatchDataset(Dataset):
    def __init__(self, data, labels):
        # data shape: (N, H, W, C) -> convert to (N, C, H, W)
        self.data   = torch.tensor(data, dtype=torch.float32).permute(0, 3, 1, 2)
        self.labels = torch.tensor(labels, dtype=torch.long)
    def __len__(self):
        return len(self.data)
    def __getitem__(self, idx):
        return self.data[idx], self.labels[idx]

# ============================================================
# ResNet-style model for 64x64 patches
# ============================================================
class ResBlock(nn.Module):
    def __init__(self, channels):
        super().__init__()
        self.block = nn.Sequential(
            nn.Conv2d(channels, channels, 3, padding=1, bias=False),
            nn.BatchNorm2d(channels),
            nn.ReLU(inplace=True),
            nn.Conv2d(channels, channels, 3, padding=1, bias=False),
            nn.BatchNorm2d(channels),
        )
        self.relu = nn.ReLU(inplace=True)

    def forward(self, x):
        return self.relu(x + self.block(x))

class ResNet(pl.LightningModule):
    def __init__(self, num_classes=4, in_channels=16, lr=3e-4,
                 max_epochs=200, class_weights=None):
        super().__init__()
        self.lr = lr
        self.max_epochs = max_epochs

        self.encoder = nn.Sequential(
            nn.Conv2d(in_channels, 64, 3, padding=1, bias=False),
            nn.BatchNorm2d(64), nn.ReLU(inplace=True),
            ResBlock(64),
            nn.MaxPool2d(2),        # 64 -> 32
            nn.Conv2d(64, 128, 3, padding=1, bias=False),
            nn.BatchNorm2d(128), nn.ReLU(inplace=True),
            ResBlock(128),
            nn.MaxPool2d(2),        # 32 -> 16
            nn.Conv2d(128, 256, 3, padding=1, bias=False),
            nn.BatchNorm2d(256), nn.ReLU(inplace=True),
            ResBlock(256),
            nn.AdaptiveAvgPool2d((1, 1)),
        )
        self.classifier = nn.Sequential(
            nn.Flatten(),
            nn.Linear(256, 128),
            nn.ReLU(), nn.Dropout(0.3),
            nn.Linear(128, num_classes),
        )
        if class_weights is not None:
            self.register_buffer("class_weights", class_weights.float())
        else:
            self.class_weights = None

    def forward(self, x):
        return self.classifier(self.encoder(x))

    def _loss(self, logits, y):
        return F.cross_entropy(logits, y, weight=self.class_weights)

    def training_step(self, batch, _):
        x, y = batch
        loss = self._loss(self(x), y)
        self.log("train_loss", loss, prog_bar=True)
        return loss

    def validation_step(self, batch, _):
        x, y = batch
        logits = self(x)
        loss = self._loss(logits, y)
        acc = (logits.argmax(1) == y).float().mean()
        self.log("val_loss", loss, prog_bar=True)
        self.log("val_acc", acc, prog_bar=True)

    def test_step(self, batch, _):
        x, y = batch
        logits = self(x)
        loss = self._loss(logits, y)
        acc = (logits.argmax(1) == y).float().mean()
        self.log("test_loss", loss)
        self.log("test_acc", acc, prog_bar=True)

    def configure_optimizers(self):
        opt = torch.optim.AdamW(self.parameters(), lr=self.lr, weight_decay=1e-4)
        sched = torch.optim.lr_scheduler.CosineAnnealingLR(opt, T_max=self.max_epochs)
        return {"optimizer": opt, "lr_scheduler": sched}

# ============================================================
# Load data
# ============================================================
FINAL_DIR = Path("/p/scratch/training2600/TeamGudnason/training_data/final_block_64")

X_train_raw = np.load(FINAL_DIR / "patches_train.npz")['patches']
y_train_raw = np.load(FINAL_DIR / "labels_train.npy")
X_val_raw   = np.load(FINAL_DIR / "patches_val.npz")['patches']
y_val_raw   = np.load(FINAL_DIR / "labels_val.npy")
X_test_raw  = np.load(FINAL_DIR / "patches_test.npz")['patches']
y_test_raw  = np.load(FINAL_DIR / "labels_test.npy")

print(f"Patch shape: {X_train_raw.shape}")

# Normalize
X_train_raw = X_train_raw.astype(np.float32) * 0.0001
X_val_raw   = X_val_raw.astype(np.float32)   * 0.0001
X_test_raw  = X_test_raw.astype(np.float32)  * 0.0001

# Label setup -- top 4 classes
unique_all    = np.unique(y_train_raw)
label_to_idx  = {int(lbl): i for i, lbl in enumerate(unique_all)}
idx_to_corine = {i: int(lbl) for i, lbl in enumerate(unique_all)}

y_all_tr = np.array([label_to_idx[int(l)] for l in y_train_raw], dtype=np.int64)
y_all_va = np.array([label_to_idx.get(int(l), -1) for l in y_val_raw], dtype=np.int64)
y_all_te = np.array([label_to_idx.get(int(l), -1) for l in y_test_raw], dtype=np.int64)

X_val_raw,  y_all_va = X_val_raw[y_all_va != -1],   y_all_va[y_all_va != -1]
X_test_raw, y_all_te = X_test_raw[y_all_te != -1],  y_all_te[y_all_te != -1]

# Drop < 100 samples
classes_full, counts_full = np.unique(y_all_tr, return_counts=True)
valid = set(classes_full[counts_full >= 100])
mask_tr = np.isin(y_all_tr, list(valid))
mask_va = np.isin(y_all_va, list(valid))
mask_te = np.isin(y_all_te, list(valid))
X_tr, y_tr = X_train_raw[mask_tr], y_all_tr[mask_tr]
X_va, y_va = X_val_raw[mask_va],   y_all_va[mask_va]
X_te, y_te = X_test_raw[mask_te],  y_all_te[mask_te]

unique_final = np.unique(y_tr)
final_map = {int(l): i for i, l in enumerate(unique_final)}
y_tr = np.array([final_map[l] for l in y_tr], dtype=np.int64)
y_va = np.array([final_map[l] for l in y_va], dtype=np.int64)
y_te = np.array([final_map[l] for l in y_te], dtype=np.int64)

# Top 4
classes_all, counts_all = np.unique(y_tr, return_counts=True)
top4_idx = classes_all[np.argsort(counts_all)[::-1][:4]]
mask_tr = np.isin(y_tr, top4_idx)
mask_va = np.isin(y_va, top4_idx)
mask_te = np.isin(y_te, top4_idx)
X_tr, y_tr = X_tr[mask_tr], y_tr[mask_tr]
X_va, y_va = X_va[mask_va], y_va[mask_va]
X_te, y_te = X_te[mask_te], y_te[mask_te]

top4_map = {int(c): i for i, c in enumerate(sorted(top4_idx))}
y_train = np.array([top4_map[l] for l in y_tr], dtype=np.int64)
y_val   = np.array([top4_map[l] for l in y_va], dtype=np.int64)
y_test  = np.array([top4_map[l] for l in y_te], dtype=np.int64)

num_classes = 4
CLASS_NAMES = {i: f"CORINE {idx_to_corine[unique_final[int(sorted(top4_idx)[i])]]}"
               for i in range(4)}

classes_k, counts_k = np.unique(y_train, return_counts=True)
print(f"\nTrain: {X_tr.shape[0]}, Val: {X_va.shape[0]}, Test: {X_te.shape[0]}")
for c, n in zip(classes_k, counts_k):
    print(f"  {CLASS_NAMES[c]}: {n} ({100*n/len(y_train):.1f}%)")

# ============================================================
# Weighted sampler
# ============================================================
sample_weights = np.array([1.0/counts_k[l] for l in y_train], dtype=np.float64)
sampler = WeightedRandomSampler(
    weights=torch.as_tensor(sample_weights, dtype=torch.double),
    num_samples=len(sample_weights),
    replacement=True,
)

# ============================================================
# Dataloaders
# ============================================================
train_dataset = PatchDataset(X_tr, y_train)
val_dataset   = PatchDataset(X_va, y_val)
test_dataset  = PatchDataset(X_te, y_test)

train_dataloader = DataLoader(train_dataset, batch_size=64, sampler=sampler,
                               shuffle=False, num_workers=4, pin_memory=False)
val_dataloader   = DataLoader(val_dataset,   batch_size=64, shuffle=False,
                               num_workers=4, pin_memory=False)
test_dataloader  = DataLoader(test_dataset,  batch_size=64, shuffle=False,
                               num_workers=4, pin_memory=False)

# ============================================================
# Model + Trainer
# ============================================================
max_epochs = 100

model_resnet = ResNet(
    num_classes=4, in_channels=16,
    lr=3e-4, max_epochs=max_epochs,
    class_weights=None,
)

early_stop = pl.callbacks.EarlyStopping(monitor="val_loss", patience=15, mode="min")

trainer = pl.Trainer(
    accelerator="gpu", devices=1, max_epochs=max_epochs,
    gradient_clip_val=1.0, log_every_n_steps=10,
    enable_progress_bar=True, callbacks=[early_stop],
)

trainer.fit(model_resnet, train_dataloader, val_dataloader)
trainer.test(model_resnet, dataloaders=test_dataloader)

# ============================================================
# Diagnostics
# ============================================================
model_resnet.eval()
model_resnet = model_resnet.cuda()
preds_all, targets_all = [], []
with torch.no_grad():
    for xb, yb in test_dataloader:
        logits = model_resnet(xb.cuda())
        preds_all.append(logits.argmax(1).cpu().numpy())
        targets_all.append(yb.numpy())

y_true = np.concatenate(targets_all).astype(np.int64)
y_pred = np.concatenate(preds_all).astype(np.int64)

cm = np.zeros((4, 4), dtype=np.int64)
for t, p in zip(y_true, y_pred):
    cm[t, p] += 1

tp        = np.diag(cm).astype(np.float64)
support   = cm.sum(axis=1).astype(np.float64)
pred_c    = cm.sum(axis=0).astype(np.float64)
recall    = np.divide(tp, support, out=np.zeros_like(tp), where=support > 0)
precision = np.divide(tp, pred_c,  out=np.zeros_like(tp), where=pred_c > 0)
f1        = np.divide(2*precision*recall, precision+recall,
                      out=np.zeros_like(tp), where=(precision+recall) > 0)

print(f"\n========== RESULTS (ResNet, 64x64, block split) ==========")
print(f"Overall accuracy:  {tp.sum()/cm.sum():.4f}")
print(f"Balanced accuracy: {recall.mean():.4f}")
print(f"Macro-F1:          {f1.mean():.4f}")
print(f"\nPer-class:")
for i in range(4):
    print(f"  {CLASS_NAMES[i]:<20} recall={recall[i]:.3f} precision={precision[i]:.3f} f1={f1[i]:.3f} support={int(support[i])}")
print(f"\nConfusion matrix:\n{cm}")
print(f"\nPredicted distribution: {np.unique(y_pred, return_counts=True)}")

Patch shape: (10000, 64, 64, 16)

Train: 7882, Val: 1544, Test: 1546
  CORINE 2: 783 (9.9%)
  CORINE 12: 5073 (64.4%)
  CORINE 18: 524 (6.6%)
  CORINE 23: 1502 (19.1%)


GPU available: True (cuda), used: True
TPU available: False, using: 0 TPU cores
💡 Tip: For seamless cloud logging and experiment tracking, try installing [litlogger](https://pypi.org/project/litlogger/) to enable LitLogger, which logs metrics and artifacts automatically to the Lightning Experiments platform.
💡 Tip: For seamless cloud uploads and versioning, try installing [litmodels](https://pypi.org/project/litmodels/) to enable LitModelCheckpoint, which syncs automatically with the Lightning model registry.
/p/project1/training2600/TeamGudnason/envs/danube/lib/python3.12/site-packages/lightning/pytorch/callbacks/model_checkpoint.py:881: Checkpoint directory /p/project1/training2600/gudnason2/repos/EO_danube_river/lightning_logs/version_14664910/checkpoints exists and is not empty.
LOCAL_RANK: 0 - CUDA_VISIBLE_DEVICES: [0]


┏━━━┳━━━━━━━━━━━━┳━━━━━━━━━━━━┳━━━━━━━━┳━━━━━━━┳━━━━━━━┓
┃   ┃ Name       ┃ Type       ┃ Params ┃ Mode  ┃ FLOPs ┃
┡━━━╇━━━━━━━━━━━━╇━━━━━━━━━━━━╇━━━━━━━━╇━━━━━━━╇━━━━━━━┩
│ 0 │ encoder    │ Sequential │  1.9 M │ train │     0 │
│ 1 │ classifier │ Sequential │ 33.4 K │ train │     0 │
└───┴────────────┴────────────┴────────┴───────┴───────┘

Trainable params: 2.0 M                                                                                            
Non-trainable params: 0                                                                                            
Total params: 2.0 M                                                                                                
Total estimated model params size (MB): 7                                                                          
Modules in train mode: 43                                                                                          
Modules in eval mode: 0                                                                                            
Total FLOPs: 0

SLURM auto-requeueing enabled. Setting signal handlers.


Output()

LOCAL_RANK: 0 - CUDA_VISIBLE_DEVICES: [0]
SLURM auto-requeueing enabled. Setting signal handlers.


Output()

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━━━━┓
┃        Test metric        ┃       DataLoader 0        ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━━━━┩
│         test_acc          │    0.8861578106880188     │
│         test_loss         │    0.43873822689056396    │
└───────────────────────────┴───────────────────────────┘


========== RESULTS (ResNet, 64x64, block split) ==========
Overall accuracy:  0.8862
Balanced accuracy: 0.8097
Macro-F1:          0.8046

Per-class:
  CORINE 2             recall=0.811 precision=0.849 f1=0.829 support=132
  CORINE 12            recall=0.913 precision=0.941 f1=0.927 support=907
  CORINE 18            recall=0.574 precision=0.520 f1=0.545 support=115
  CORINE 23            recall=0.941 precision=0.893 f1=0.917 support=392

Confusion matrix:
[[107  12   6   7]
 [ 13 828  43  23]
 [  4  31  66  14]
 [  2   9  12 369]]

Predicted distribution: (array([0, 1, 2, 3]), array([126, 880, 127, 413]))


In [18]:
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import numpy as np

# Confusion matrix from ResNet results
cm = np.array([
    [107,  12,   6,   7],
    [ 13, 828,  43,  23],
    [  4,  31,  66,  14],
    [  2,   9,  12, 369]
])

class_names = ['Urban\n(CORINE 2)', 'Arable\n(CORINE 12)', 
               'Pastures\n(CORINE 18)', 'Forest\n(CORINE 23)']

# Normalize by row
cm_norm = cm.astype('float') / cm.sum(axis=1, keepdims=True)

fig, ax = plt.subplots(figsize=(7, 6))
im = ax.imshow(cm_norm, cmap='Blues', vmin=0, vmax=1)
plt.colorbar(im, ax=ax, label='Recall')

ax.set_xticks(range(4))
ax.set_yticks(range(4))
ax.set_xticklabels(class_names, fontsize=9)
ax.set_yticklabels(class_names, fontsize=9)
ax.set_xlabel('Predicted label')
ax.set_ylabel('True label')
ax.set_title('Confusion Matrix (ResNet 64x64, block cross-validation)')

for i in range(4):
    for j in range(4):
        color = 'white' if cm_norm[i, j] > 0.5 else 'black'
        ax.text(j, i, f'{cm_norm[i,j]:.2f}\n({cm[i,j]})',
                ha='center', va='center', fontsize=8, color=color)

plt.tight_layout()
plt.savefig('/p/scratch/training2600/TeamGudnason/results/confusion_matrix.png', 
            dpi=150, bbox_inches='tight')
print("Vistað.")

Vistað.


In [ ]:
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import numpy as np

# Class distribution in full dataset (from our data exploration)
classes = ['Urban\n(CORINE 2)', 'Pastures\n(CORINE 18)', 
           'Water\n(CORINE 41)', 'Forest\n(CORINE 23)', 
           'Arable land\n(CORINE 12)']
percentages = [4.8, 5.5, 6.1, 20.3, 45.0]
colors = ['#4878CF', '#6ACC65', '#D65F5F', '#B47CC7', '#C4AD66']

fig, ax = plt.subplots(figsize=(8, 5))
bars = ax.barh(classes, percentages, color=colors, edgecolor='white', height=0.6)

# Add percentage labels
for bar, pct in zip(bars, percentages):
    ax.text(bar.get_width() + 0.5, bar.get_y() + bar.get_height()/2,
            f'{pct:.1f}%', va='center', fontsize=10)

ax.set_xlabel('Share of labeled pixels (%)')
ax.set_title('Class distribution in study area (MGRS tile T33TYN)')
ax.set_xlim(0, 52)
ax.spines['top'].set_visible(False)
ax.spines['right'].set_visible(False)

plt.tight_layout()
plt.savefig('/p/scratch/training2600/TeamGudnason/results/class_distribution.png',
            dpi=150, bbox_inches='tight')
print("Vistað.")

In [ ]:
## þjalfa model aftur fyrr ground truth

In [3]:
import os
os.environ["CUDA_LAUNCH_BLOCKING"] = "1"
os.environ["CUDA_VISIBLE_DEVICES"] = "0"

import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
import lightning.pytorch as pl
from torch.utils.data import DataLoader, Dataset, WeightedRandomSampler
from pathlib import Path

# ============================================================
# Dataset
# ============================================================
class PatchDataset(Dataset):
    def __init__(self, data, labels):
        # data shape: (N, H, W, C) -> convert to (N, C, H, W)
        self.data   = torch.tensor(data, dtype=torch.float32).permute(0, 3, 1, 2)
        self.labels = torch.tensor(labels, dtype=torch.long)
    def __len__(self):
        return len(self.data)
    def __getitem__(self, idx):
        return self.data[idx], self.labels[idx]

# ============================================================
# ResNet-style model for 64x64 patches
# ============================================================
class ResBlock(nn.Module):
    def __init__(self, channels):
        super().__init__()
        self.block = nn.Sequential(
            nn.Conv2d(channels, channels, 3, padding=1, bias=False),
            nn.BatchNorm2d(channels),
            nn.ReLU(inplace=True),
            nn.Conv2d(channels, channels, 3, padding=1, bias=False),
            nn.BatchNorm2d(channels),
        )
        self.relu = nn.ReLU(inplace=True)

    def forward(self, x):
        return self.relu(x + self.block(x))

class ResNet(pl.LightningModule):
    def __init__(self, num_classes=4, in_channels=16, lr=3e-4,
                 max_epochs=200, class_weights=None):
        super().__init__()
        self.lr = lr
        self.max_epochs = max_epochs

        self.encoder = nn.Sequential(
            nn.Conv2d(in_channels, 64, 3, padding=1, bias=False),
            nn.BatchNorm2d(64), nn.ReLU(inplace=True),
            ResBlock(64),
            nn.MaxPool2d(2),        # 64 -> 32
            nn.Conv2d(64, 128, 3, padding=1, bias=False),
            nn.BatchNorm2d(128), nn.ReLU(inplace=True),
            ResBlock(128),
            nn.MaxPool2d(2),        # 32 -> 16
            nn.Conv2d(128, 256, 3, padding=1, bias=False),
            nn.BatchNorm2d(256), nn.ReLU(inplace=True),
            ResBlock(256),
            nn.AdaptiveAvgPool2d((1, 1)),
        )
        self.classifier = nn.Sequential(
            nn.Flatten(),
            nn.Linear(256, 128),
            nn.ReLU(), nn.Dropout(0.3),
            nn.Linear(128, num_classes),
        )
        if class_weights is not None:
            self.register_buffer("class_weights", class_weights.float())
        else:
            self.class_weights = None

    def forward(self, x):
        return self.classifier(self.encoder(x))

    def _loss(self, logits, y):
        return F.cross_entropy(logits, y, weight=self.class_weights)

    def training_step(self, batch, _):
        x, y = batch
        loss = self._loss(self(x), y)
        self.log("train_loss", loss, prog_bar=True)
        return loss

    def validation_step(self, batch, _):
        x, y = batch
        logits = self(x)
        loss = self._loss(logits, y)
        acc = (logits.argmax(1) == y).float().mean()
        self.log("val_loss", loss, prog_bar=True)
        self.log("val_acc", acc, prog_bar=True)

    def test_step(self, batch, _):
        x, y = batch
        logits = self(x)
        loss = self._loss(logits, y)
        acc = (logits.argmax(1) == y).float().mean()
        self.log("test_loss", loss)
        self.log("test_acc", acc, prog_bar=True)

    def configure_optimizers(self):
        opt = torch.optim.AdamW(self.parameters(), lr=self.lr, weight_decay=1e-4)
        sched = torch.optim.lr_scheduler.CosineAnnealingLR(opt, T_max=self.max_epochs)
        return {"optimizer": opt, "lr_scheduler": sched}

# ============================================================
# Load data
# ============================================================
FINAL_DIR = Path("/p/scratch/training2600/TeamGudnason/training_data/final_block_64")

X_train_raw = np.load(FINAL_DIR / "patches_train.npz")['patches']
y_train_raw = np.load(FINAL_DIR / "labels_train.npy")
X_val_raw   = np.load(FINAL_DIR / "patches_val.npz")['patches']
y_val_raw   = np.load(FINAL_DIR / "labels_val.npy")
X_test_raw  = np.load(FINAL_DIR / "patches_test.npz")['patches']
y_test_raw  = np.load(FINAL_DIR / "labels_test.npy")

print(f"Patch shape: {X_train_raw.shape}")

# Normalize
X_train_raw = X_train_raw.astype(np.float32) * 0.0001
X_val_raw   = X_val_raw.astype(np.float32)   * 0.0001
X_test_raw  = X_test_raw.astype(np.float32)  * 0.0001

# Label setup -- top 4 classes
unique_all    = np.unique(y_train_raw)
label_to_idx  = {int(lbl): i for i, lbl in enumerate(unique_all)}
idx_to_corine = {i: int(lbl) for i, lbl in enumerate(unique_all)}

y_all_tr = np.array([label_to_idx[int(l)] for l in y_train_raw], dtype=np.int64)
y_all_va = np.array([label_to_idx.get(int(l), -1) for l in y_val_raw], dtype=np.int64)
y_all_te = np.array([label_to_idx.get(int(l), -1) for l in y_test_raw], dtype=np.int64)

X_val_raw,  y_all_va = X_val_raw[y_all_va != -1],   y_all_va[y_all_va != -1]
X_test_raw, y_all_te = X_test_raw[y_all_te != -1],  y_all_te[y_all_te != -1]

# Drop < 100 samples
classes_full, counts_full = np.unique(y_all_tr, return_counts=True)
valid = set(classes_full[counts_full >= 100])
mask_tr = np.isin(y_all_tr, list(valid))
mask_va = np.isin(y_all_va, list(valid))
mask_te = np.isin(y_all_te, list(valid))
X_tr, y_tr = X_train_raw[mask_tr], y_all_tr[mask_tr]
X_va, y_va = X_val_raw[mask_va],   y_all_va[mask_va]
X_te, y_te = X_test_raw[mask_te],  y_all_te[mask_te]

unique_final = np.unique(y_tr)
final_map = {int(l): i for i, l in enumerate(unique_final)}
y_tr = np.array([final_map[l] for l in y_tr], dtype=np.int64)
y_va = np.array([final_map[l] for l in y_va], dtype=np.int64)
y_te = np.array([final_map[l] for l in y_te], dtype=np.int64)

# Top 4
classes_all, counts_all = np.unique(y_tr, return_counts=True)
top4_idx = classes_all[np.argsort(counts_all)[::-1][:4]]
mask_tr = np.isin(y_tr, top4_idx)
mask_va = np.isin(y_va, top4_idx)
mask_te = np.isin(y_te, top4_idx)
X_tr, y_tr = X_tr[mask_tr], y_tr[mask_tr]
X_va, y_va = X_va[mask_va], y_va[mask_va]
X_te, y_te = X_te[mask_te], y_te[mask_te]

top4_map = {int(c): i for i, c in enumerate(sorted(top4_idx))}
y_train = np.array([top4_map[l] for l in y_tr], dtype=np.int64)
y_val   = np.array([top4_map[l] for l in y_va], dtype=np.int64)
y_test  = np.array([top4_map[l] for l in y_te], dtype=np.int64)

num_classes = 4
CLASS_NAMES = {i: f"CORINE {idx_to_corine[unique_final[int(sorted(top4_idx)[i])]]}"
               for i in range(4)}

classes_k, counts_k = np.unique(y_train, return_counts=True)
print(f"\nTrain: {X_tr.shape[0]}, Val: {X_va.shape[0]}, Test: {X_te.shape[0]}")
for c, n in zip(classes_k, counts_k):
    print(f"  {CLASS_NAMES[c]}: {n} ({100*n/len(y_train):.1f}%)")

# ============================================================
# Weighted sampler
# ============================================================
sample_weights = np.array([1.0/counts_k[l] for l in y_train], dtype=np.float64)
sampler = WeightedRandomSampler(
    weights=torch.as_tensor(sample_weights, dtype=torch.double),
    num_samples=len(sample_weights),
    replacement=True,
)

# ============================================================
# Dataloaders
# ============================================================
train_dataset = PatchDataset(X_tr, y_train)
val_dataset   = PatchDataset(X_va, y_val)
test_dataset  = PatchDataset(X_te, y_test)

train_dataloader = DataLoader(train_dataset, batch_size=64, sampler=sampler,
                               shuffle=False, num_workers=4, pin_memory=False)
val_dataloader   = DataLoader(val_dataset,   batch_size=64, shuffle=False,
                               num_workers=4, pin_memory=False)
test_dataloader  = DataLoader(test_dataset,  batch_size=64, shuffle=False,
                               num_workers=4, pin_memory=False)

# ============================================================
# Model + Trainer
# ============================================================
max_epochs = 100

model_resnet = ResNet(
    num_classes=4, in_channels=16,
    lr=3e-4, max_epochs=max_epochs,
    class_weights=None,
)

early_stop = pl.callbacks.EarlyStopping(monitor="val_loss", patience=15, mode="min")

trainer = pl.Trainer(
    accelerator="gpu", devices=1, max_epochs=max_epochs,
    gradient_clip_val=1.0, log_every_n_steps=10,
    enable_progress_bar=True, callbacks=[early_stop],
)

trainer.fit(model_resnet, train_dataloader, val_dataloader)
trainer.test(model_resnet, dataloaders=test_dataloader)

# ============================================================
# Diagnostics
# ============================================================
model_resnet.eval()
model_resnet = model_resnet.cuda()
preds_all, targets_all = [], []
with torch.no_grad():
    for xb, yb in test_dataloader:
        logits = model_resnet(xb.cuda())
        preds_all.append(logits.argmax(1).cpu().numpy())
        targets_all.append(yb.numpy())

y_true = np.concatenate(targets_all).astype(np.int64)
y_pred = np.concatenate(preds_all).astype(np.int64)

cm = np.zeros((4, 4), dtype=np.int64)
for t, p in zip(y_true, y_pred):
    cm[t, p] += 1

tp        = np.diag(cm).astype(np.float64)
support   = cm.sum(axis=1).astype(np.float64)
pred_c    = cm.sum(axis=0).astype(np.float64)
recall    = np.divide(tp, support, out=np.zeros_like(tp), where=support > 0)
precision = np.divide(tp, pred_c,  out=np.zeros_like(tp), where=pred_c > 0)
f1        = np.divide(2*precision*recall, precision+recall,
                      out=np.zeros_like(tp), where=(precision+recall) > 0)

print(f"\n========== RESULTS (ResNet, 64x64, block split) ==========")
print(f"Overall accuracy:  {tp.sum()/cm.sum():.4f}")
print(f"Balanced accuracy: {recall.mean():.4f}")
print(f"Macro-F1:          {f1.mean():.4f}")
print(f"\nPer-class:")
for i in range(4):
    print(f"  {CLASS_NAMES[i]:<20} recall={recall[i]:.3f} precision={precision[i]:.3f} f1={f1[i]:.3f} support={int(support[i])}")
print(f"\nConfusion matrix:\n{cm}")
print(f"\nPredicted distribution: {np.unique(y_pred, return_counts=True)}")

Patch shape: (10000, 64, 64, 16)

Train: 7882, Val: 1544, Test: 1546
  CORINE 2: 783 (9.9%)
  CORINE 12: 5073 (64.4%)
  CORINE 18: 524 (6.6%)
  CORINE 23: 1502 (19.1%)


GPU available: True (cuda), used: True
TPU available: False, using: 0 TPU cores
💡 Tip: For seamless cloud logging and experiment tracking, try installing [litlogger](https://pypi.org/project/litlogger/) to enable LitLogger, which logs metrics and artifacts automatically to the Lightning Experiments platform.
💡 Tip: For seamless cloud uploads and versioning, try installing [litmodels](https://pypi.org/project/litmodels/) to enable LitModelCheckpoint, which syncs automatically with the Lightning model registry.
LOCAL_RANK: 0 - CUDA_VISIBLE_DEVICES: [0]


┏━━━┳━━━━━━━━━━━━┳━━━━━━━━━━━━┳━━━━━━━━┳━━━━━━━┳━━━━━━━┓
┃   ┃ Name       ┃ Type       ┃ Params ┃ Mode  ┃ FLOPs ┃
┡━━━╇━━━━━━━━━━━━╇━━━━━━━━━━━━╇━━━━━━━━╇━━━━━━━╇━━━━━━━┩
│ 0 │ encoder    │ Sequential │  1.9 M │ train │     0 │
│ 1 │ classifier │ Sequential │ 33.4 K │ train │     0 │
└───┴────────────┴────────────┴────────┴───────┴───────┘

Trainable params: 2.0 M                                                                                            
Non-trainable params: 0                                                                                            
Total params: 2.0 M                                                                                                
Total estimated model params size (MB): 7                                                                          
Modules in train mode: 43                                                                                          
Modules in eval mode: 0                                                                                            
Total FLOPs: 0

SLURM auto-requeueing enabled. Setting signal handlers.


Output()

LOCAL_RANK: 0 - CUDA_VISIBLE_DEVICES: [0]
SLURM auto-requeueing enabled. Setting signal handlers.


Output()

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━━━━┓
┃        Test metric        ┃       DataLoader 0        ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━━━━┩
│         test_acc          │    0.8699870705604553     │
│         test_loss         │    0.49500906467437744    │
└───────────────────────────┴───────────────────────────┘


========== RESULTS (ResNet, 64x64, block split) ==========
Overall accuracy:  0.8700
Balanced accuracy: 0.8351
Macro-F1:          0.8004

Per-class:
  CORINE 2             recall=0.902 precision=0.768 f1=0.829 support=132
  CORINE 12            recall=0.861 precision=0.964 f1=0.910 support=907
  CORINE 18            recall=0.626 precision=0.522 f1=0.569 support=115
  CORINE 23            recall=0.952 precision=0.842 f1=0.893 support=392

Confusion matrix:
[[119   4   2   7]
 [ 32 781  51  43]
 [  3  20  72  20]
 [  1   5  13 373]]

Predicted distribution: (array([0, 1, 2, 3]), array([155, 810, 138, 443]))


In [4]:
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import numpy as np

# Color map -- accessible colors
COLORS = {
    0: '#4878CF',  # Urban -- blár
    1: '#C4AD66',  # Arable -- gulbrúnn
    2: '#6ACC65',  # Pastures -- grænn
    3: '#2E8B57',  # Forest -- dökkgrænn
}

# Taka 10 patches úr test set
rng = np.random.RandomState(42)
n_show = 10
indices = rng.choice(len(y_true), n_show, replace=False)

fig, axes = plt.subplots(2, n_show, figsize=(18, 4))

for col, idx in enumerate(indices):
    true_label = y_true[idx]
    pred_label = y_pred[idx]

    # RGB mynd -- August acquisition (bands 8,9,10 = B02,B03,B04)
    patch = X_te[idx]  # (64, 64, 16)
    rgb = patch[:, :, 8:11].copy()
    rgb = (rgb - rgb.min()) / (rgb.max() - rgb.min() + 1e-6)
    rgb = rgb[:, :, ::-1]  # BGR -> RGB

    # Ground truth
    axes[0, col].imshow(rgb)
    axes[0, col].set_title(CLASS_NAMES[true_label], fontsize=7, 
                            color=COLORS[true_label], fontweight='bold')
    axes[0, col].axis('off')

    # Prediction -- color block með border
    pred_color = [int(COLORS[pred_label][i:i+2], 16)/255 for i in (1, 3, 5)]
    pred_img = np.full((64, 64, 3), pred_color)
    axes[1, col].imshow(pred_img)
    axes[1, col].set_title(CLASS_NAMES[pred_label], fontsize=7,
                            color=COLORS[pred_label], fontweight='bold')
    
    # Rauður border ef rangt, grænn ef rétt
    border_color = '#2E8B57' if true_label == pred_label else '#D65F5F'
    for spine in axes[1, col].spines.values():
        spine.set_edgecolor(border_color)
        spine.set_linewidth(3)
    axes[1, col].axis('off')

# Row labels
axes[0, 0].set_ylabel('Ground truth\n(RGB patch)', fontsize=8)
axes[1, 0].set_ylabel('Prediction\n(color block)', fontsize=8)

# Legend
patches = [mpatches.Patch(color=COLORS[i], label=CLASS_NAMES[i]) for i in range(4)]
fig.legend(handles=patches, loc='lower center', ncol=4, fontsize=9,
           bbox_to_anchor=(0.5, -0.02))

plt.suptitle('Sample predictions vs. ground truth\n(green border = correct, orange border = incorrect)',
             fontsize=10)
plt.tight_layout()
plt.savefig('/p/scratch/training2600/TeamGudnason/results/prediction_samples.png',
            dpi=150, bbox_inches='tight')
print("Vistað.")

Vistað.


In [5]:
import rasterio
with rasterio.open('/p/scratch/training2600/TeamGudnason/data/aligned_data/S2B_MSIL2A_20180408T095029_N0500_R079_T33TYN_20230906T162829_stacked.tif') as src:
    bounds = src.bounds
    crs = src.crs
    print(f"Bounds: {bounds}")
    print(f"CRS: {crs}")

Bounds: BoundingBox(left=699960.0, bottom=5190240.0, right=809760.0, top=5300040.0)
CRS: EPSG:32633


In [6]:
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
from pyproj import Transformer
import numpy as np

# Convert UTM bounds to WGS84
transformer = Transformer.from_crs("EPSG:32633", "EPSG:4326", always_xy=True)

left, bottom = 699960.0, 5190240.0
right, top   = 809760.0, 5300040.0

# Corner points
corners_utm = [(left, bottom), (right, bottom), (right, top), (left, top), (left, bottom)]
corners_wgs = [transformer.transform(x, y) for x, y in corners_utm]
lons = [c[0] for c in corners_wgs]
lats = [c[1] for c in corners_wgs]

print(f"Lon: {min(lons):.2f} - {max(lons):.2f}")
print(f"Lat: {min(lats):.2f} - {max(lats):.2f}")
center_lon = (min(lons) + max(lons)) / 2
center_lat = (min(lats) + max(lats)) / 2
print(f"Center: {center_lon:.2f}, {center_lat:.2f}")

# Plot
fig, ax = plt.subplots(figsize=(6, 6))

ax.fill(lons, lats, color='#f0e8d0', alpha=0.8)
ax.plot(lons, lats, 'k-', linewidth=1.5)

# Danube approximate path
danube_lon = [min(lons), min(lons)+0.3, min(lons)+0.6, min(lons)+0.9, max(lons)]
danube_lat = [max(lats)-0.2, max(lats)-0.3, max(lats)-0.4, max(lats)-0.5, max(lats)-0.6]
ax.plot(danube_lon, danube_lat, color='#4878CF', linewidth=2, label='Danube river')

# Mark center
ax.plot(center_lon, center_lat, 'r*', markersize=12, label='Tile center')

ax.set_xlabel('Longitude')
ax.set_ylabel('Latitude')
ax.set_title(f'Study area: MGRS tile T33TYN\n(Danube basin, Central Europe)')
ax.legend(loc='lower right', fontsize=8)
ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig('/p/scratch/training2600/TeamGudnason/results/study_area_map.png',
            dpi=150, bbox_inches='tight')
print("Vistað.")

Lon: 17.62 - 19.14
Lat: 46.79 - 47.82
Center: 18.38, 47.31
Vistað.
